waibuyanzheng

In [ ]:
import pandas as pd

# 输入输出文件路径
input_file = f"C:/Users/User/Desktop/pythondata/预后模型评估/原始数据/增加的新的数据.xlsx"
output_file = f"C:/Users/User/Desktop/pythondata/预后模型评估/zs/output1增加的新的数据.xlsx"

# 读取Excel文件
df = pd.read_excel(input_file)

# 初始化一个空列表用于存储结果
result = []

# 定义处理每个 PET 组的函数
def process_pet_group(row, pet_prefix):
    treatment_col = f'{pet_prefix}检查号'
    if pd.notna(row[treatment_col]):
        new_row = {
            '姓名': row['姓名'],
            '病历号': row['病历号'],#病历号
            '检查号': row[treatment_col],
            '治疗前后/复发': row[f'{pet_prefix}治疗前后/复发（1/2/3）'],
            '检查时间': row[f'{pet_prefix}检查时间'],
            '影像表现': row[f'{pet_prefix}影像表现'],
            '诊断结论': row[f'{pet_prefix}诊断结论']
        }
        result.append(new_row)

# 遍历每一行
for _, row in df.iterrows():
    process_pet_group(row, 'PET1')
    process_pet_group(row, 'PET2')
    process_pet_group(row, 'PET3')
    process_pet_group(row, 'PET4')
    process_pet_group(row, 'PET5')
    process_pet_group(row, 'PET6')

# 转换为DataFrame并写入新文件
result_df = pd.DataFrame(result)
result_df.to_excel(output_file, index=False)

print(f"已成功提取数据到文件：{output_file}")

In [3]:
#deepseek qwenplus
import pandas as pd
import json
import re

# 配置参数 内部外部
#excel_file = r"C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\匹配及关联数据.xlsx"         # Excel 文件路径
excel_file = f"C:/Users/User/Desktop/pythondata/预后模型评估/zs/output1增加的新的数据.xlsx"        # Excel 文件路径

sheet_name = "Sheet1"             # 工作表名称
      # 输出文件路径
col1 = "影像表现"                 # 第一列列名
col2 = "诊断结论"                   # 第二列列名


# 读取 Excel 数据
df = pd.read_excel(excel_file, sheet_name=sheet_name)
# with open("xmoutput.md", "r", encoding="utf-8") as file:
#     markdown_content = file.read()  

markdown_content =  """
下面是PET-CT报告的专业术语定义用来学习参考。

扫描区包括四肢，脊柱，颅骨。
在后面的填写规范中NA也表示不确定、不明，X4和NA都可指示多发

阳性信息  填写规范：阳性/阴性
PET阳性（PET-positive）的定义
满足以下任一条，即判定为阳性病灶：
1.局灶性骨病变：
  a.在基线或随访扫描中，出现 局灶性 FDG 摄取，且其 SUVmax ≥ 纵隔血池（mediastinal blood pool SUV）或 ≥ 周围正常骨髓组织；
  b.可伴或不伴 CT 可见的溶骨性破坏。
2.髓外病变（EM）或髓旁病灶(PM)：
  a.任何软组织或骨旁区域出现 新发或持续的高代谢灶（SUVmax 标准同上）。
3.弥漫性骨髓高代谢：
  a.骨髓弥漫性 FDG 摄取高于纵隔血池，且无其他可解释原因（如感染、骨折）
PET阴性（PET-negative）的定义
必须 同时满足 以下所有条件：
1.完全消失：
  a.与基线相比，所有先前的高代谢病灶（包括局灶性骨病变、EMD、弥漫性骨髓摄取） 完全消失；
2.或代谢降至背景以下：
  a.残余灶的SUVmax < 纵隔血池 SUV 或 < 周围正常骨髓组织；
3.无新发病灶：
  a.无新发局灶性或弥漫性高代谢灶；
4.无新骨质破坏：
  a.CT部分未见新发溶骨性病变。
对于初诊患者,PET-CT阴性指的是不满足PET-CT阳性任一条件。

骨髓/骨骼整体代谢活性 填写规范：是/否 数值 是/否
骨髓/骨骼整体代谢活性增高，主要看SUV值>肝脏代谢，通常以肝脏代谢活性为参照标准。需排除化疗后骨髓增生反应或生长因子使用导致的骨髓高代谢（可能表现为弥漫性摄取增高）。

骨质破坏 填写规范：有/无
骨质破坏指的是PET-CT中显示明确的溶骨性病变（如骨皮质缺损、溶骨性破坏、穿凿样改变）。


按 MM 专用 Deauville 5 级标准： 
1 分：病灶处完全无摄取，或肉眼不可辨。
2 分：可见摄取，但病灶 SUVmax ≤ 肝脏 SUVmax。
3 分：病灶 SUVmax ＞ 肝脏，但增幅 ≤ 10 %。
4 分：病灶 SUVmax ＞ 肝脏 +10 %，但未达 2 倍。
5 分：病灶 SUVmax ≥ 2× 肝脏，或出现任何新的高摄取灶。
放射性摄取增高指的是比周围骨髓组织高，或高于肝脏基础摄取值。

病变类别有局灶性病灶（FLs），髓外病灶(EM)，髓旁病灶(PM)，溶骨性病变(L)。
局灶性病灶（FLs）填写规范：S/SP/Ex-Sp X1/X2/X3/X4 X1/X2/X3/X4 数值 数值 
局灶性病变部位分为S (Skull) SP (spine) Ex-Sp (extra-spine)，局灶性病变数量分为X1 (None) 、X2 (N =1 to 3)、 X3 (N =4 to 10)、 X4 (N >10),填写时应填写X1或X2或X3或X4。局灶性病变定义为在至少 2 个连续的 PET 切片上或大小大于5mm可见的 F-FDG 摄取增加的局灶区域，摄取增加指的是SUVmax值大于周围骨髓/骨骼组织或高于肝脏组织或大于2.5，需表现为局灶性（非弥漫性），即单个或多个离散的高摄取灶。需排除弥漫性摄取和生理学摄取（炎性摄取）。溶骨性病变是指通过CT成像检测到的骨质破坏，表现为骨皮质或骨小梁的缺失，通常提示骨髓瘤细胞浸润导致的骨吸收和骨结构破坏。
髓外病灶(EM) 填写规范： 文本 N或EN或N/EN X1/X2/X3/X4 X1/X2/X3/X4 数值
髓外病变（EM）指的是软组织/器官高摄取灶，髓外病变（EM）分为N(淋巴结)/EN(非淋巴结)。髓外病变淋巴结如果为高摄取灶，但诊断结论为随访、排除炎症、其他肿瘤性病变等则考虑髓外病变；对于非淋巴结组织，若诊断结论仅为随访考虑不是髓外病变。髓外病变（EM）部位填写时应填写N或EN或N/EN，当部位填写N/EN时数量可填写如X4/X2形式。
髓旁病灶(PM) 填写规范： 文本 X1/X2/X3/X4 X1/X2/X3/X4 数值 数值
髓旁病灶(PM)是指软组织肿块从骨髓生长到周围组织。髓外病变和髓旁病灶需要排除退行性病变、炎性病变以及非多发性骨髓瘤的肿瘤病变（影像表现和诊断结论作为依据）。
特别注意旁髓病变，临近骨且形成软组织肿块。

溶骨性病变(L) :溶骨性病变是指通过CT成像检测到的骨质破坏，表现为骨皮质或骨小梁的缺失，通常提示骨髓瘤细胞浸润导致的骨吸收和骨结构破坏。

SUVmax最大值 填写规范：数值 
SUVmax最大值指的是骨髓瘤病变，包括弥漫性病变，局灶性病变，髓外病变，髓旁病变。

骨折 填写规范：有/无 S/SP/Ex-Sp 新发/陈旧 是/否
骨折指CT病理性骨折，骨折部位分为S (Skull) SP (spine) Ex-Sp (extra-spine)

骨骼系统外科手术证据 填写规范：有/无 文本 a/b/c/d
骨骼系统外科手术证据影像学报告中需明确标注 既往或近期骨骼系统手术痕迹，其类型分为：
  a.内固定植入物（如钢板、螺钉、髓内钉）
  b.椎体成形术/后凸成形术（骨水泥填充）
  c.骨移植或人工关节置换
  d.其他手术相关结构改变（如截骨术、刮除术痕迹）

所有数据如果是无法判断或无法获取填写规范即写NA

最终输出结构化表格17行6列（待填写）如下：
| 阳性信息                     | PET阳性/阴性   |            |                       |                    |              |
|-----------------------------|----------------|------------|-----------------------|--------------------|--------------|
|                             |  （待填写）      |            |                       |                    |              |
| 骨髓/骨骼整体代谢活性         | 是否增高       | SUVmax值   | 长骨的高代谢（是/否）   |                    |              |
|                             |    （待填写）    | （待填写）   | （待填写）           |                    |              |
| 骨质破坏                     | 有/无          |            |                       |                    |              |
|                             |  （待填写）      |            |                       |                    |              |
| 病变类别                     | 部位           | 数量         | 溶骨性病变数量         | Deauville评分      | SUVmax值     |
| 局灶性病灶（FLs）            |  （待填写）     |   （待填写）    |（待填写）           | （待填写）         |  （待填写）     |
| 髓外病灶(EM)                 |  （待填写）      |  （待填写）   |  （待填写）           |  （待填写）         |  （待填写）      |
| 髓旁病灶(PM)                |   （待填写）     |  （待填写）   | （待填写）            |  （待填写）           |  （待填写）     |
|                             | SUVmax最大值   |            |                      |                      |            |
| SUVmax最大值                |  （待填写）   |             |                       |                      |            |
|                            | 有/无           | 部位        | 新发/陈旧              | 多发性骨髓瘤是否引起  |              |   
| 骨折                        | （待填写）     |  （待填写）   |   （待填写）        |  （待填写）        |             |
|                            | 有/无           | 部位        | 类型                  |                     |              |   
| 骨骼系统外科手术证据        | （待填写）       |   （待填写）    |  （待填写）           |                      |             |

真实示例3例：
a
| 阳性信息                     | PET阳性/阴性   |            |                       |                    |              |
|-----------------------------|----------------|------------|-----------------------|--------------------|--------------|
|                             | 阳性         |            |                       |                    |              |
| 骨髓/骨骼整体代谢活性         | 是否增高       | SUVmax值   | 长骨的高代谢（是/否）   |                    |              |
|                             | 是             | 2.7        | 是                    |                    |              |
| 骨质破坏                     | 有/无          |            |                       |                    |              |
|                             | 有             |            |                       |                    |              |
| 病变类别                     | 部位           | 数量         | 溶骨性病变数量         | Deauville评分      | SUVmax值     |
| 局灶性病灶（FLs）            | S/SP/Ex-Sp | X4        | NA                    | NA                 | 9.8          |
| 髓外病灶(EM)                 | N/EN           | X4/X2         | NA                    | NA                 | 3.6          |
| 髓旁病灶(PM)                | 右侧股骨       | 1          | NA                    | NA                 | 6.6          |
|                             | SUVmax最大值   |            |                      |                      |            |
| SUVmax最大值                | 9.8          |             |                       |                      |            |
|                            | 有/无           | 部位        | 新发/陈旧              | 多发性骨髓瘤是否引起  |              |   
| 骨折                        | 无             | NA         | NA                    | NA                 |             |
|                            | 有/无           | 部位        | 类型                  |                     |              |   
| 骨骼系统外科手术证据        | 无             | NA         | NA                    |                      |             |

b
| 阳性信息                     | PET阳性/阴性   |            |                       |                    |              |
|-----------------------------|----------------|------------|-----------------------|--------------------|--------------|
|                             | 阳性         |            |                       |                    |              |
| 骨髓/骨骼整体代谢活性         | 是否增高       | SUVmax值   | 长骨的高代谢（是/否）   |                    |              |
|                             | 是             | 2.7        | 是                    |                    |              |
| 骨质破坏                     | 有/无          |            |                       |                    |              |
|                             | 有             |            |                       |                    |              |
| 病变类别                     | 部位           | 数量         | 溶骨性病变数量         | Deauville评分      | SUVmax值     |
| 局灶性病灶（FLs）            | S/SP/Ex-Sp | X4        | NA                    | NA                 | 9.8          |
| 髓外病灶(EM)                 | N/EN           | X4/X2         | NA                    | NA                 | 3.6          |
| 髓旁病灶(PM)                | 右侧股骨       | 1          | NA                    | NA                 | 6.6          |
|                             | SUVmax最大值   |            |                      |                      |            |
| SUVmax最大值                | 9.8          |             |                       |                      |            |
|                            | 有/无           | 部位        | 新发/陈旧              | 多发性骨髓瘤是否引起  |              |   
| 骨折                        | 无             | NA         | NA                    | NA                 |             |
|                            | 有/无           | 部位        | 类型                  |                     |              |   
| 骨骼系统外科手术证据        | 无             | NA         | NA                    |                      |             |

c
| 阳性信息                     | PET阳性/阴性   |            |                       |                    |              |
| ---------------------------- | -------------- | ---------- | --------------------- | ------------------ | ------------ |
|                             | 阳性         |            |                       |                    |              |
| 骨髓/骨骼整体代谢活性         | 是否增高       | SUVmax值   | 长骨的高代谢（是/否）   |                    |              |
|                             | 是             | 2.7        | 是                    |                    |              |
| 骨质破坏                     | 有/无          |            |                       |                    |              |
|                             | 无             |            |                       |                    |              |
| 病变类别                     | 部位           | 数量         | 溶骨性病变数量         | Deauville评分      | SUVmax值     |
| 局灶性病灶（FLs）            | NA             | X1         | 0                     | NA                 | NA           |
| 髓外病灶(EM)                 | NA             | NA         | NA                    | NA                 | NA           |
| 髓旁病灶(PM)                | NA             | NA         | NA                    | NA                 | NA           |
|                             | SUVmax最大值   |            |                       |                    |              |
| SUVmax最大值                | 2.7            |            |                       |                    |              |
|                            | 有/无           | 部位        | 新发/陈旧              | 多发性骨髓瘤是否引起  |              |   
| 骨折                        | 无             | NA         | NA                    | NA                 |              |
|                            | 有/无           | 部位        | 类型                  |                     |              |   
| 骨骼系统外科手术证据        | 无             | NA         | NA                    |                     |              |
"""
# 生成 JSON 数据
# def process_row(row):
#     return f"{delimiter1}{row[col1]}{delimiter2}{row[col2]}" + "请生成对应的且格式一致的示例输出表格。"
mark =  "你是放射科专家，擅长进行PET-CT报告解读。"
json_obj = []
for idx, row in df.iterrows():
        # 拼接两列内容
    user_content = f"PET-CT报告如下\n" + f"影像表现：{row[col1]}  诊断结论：{row[col2]}" + "多发性骨髓瘤PET-CT自由报告如上所述" + "/n我的请求是将结构化表格（待填写）部分填写完整，并严格按照其格式输出，仅输出17行6列的结构化表格。"
        # 构建 JSON 对象
    json_obj.append({
        "custom_id": str(idx + 1),
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "qwen3",
            "messages":[
                {"role": "user", "content": mark + markdown_content + user_content}
            ],"temperature": 0.7, "top_p": 0.9}
    })
        
        # 写入文件（JSON Lines 格式）
    with open("loraqwen3zenjiashuju结构化表格输入prompt.jsonl", "w", encoding="utf-8") as f:    
        for item in json_obj:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"生成完成，文件已保存至 ")

生成完成，文件已保存至 


In [1]:
#bingzaomingxi
#deepseek qwenplus
import pandas as pd
import json
import re

# 配置参数 内部外部
#excel_file = r"C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\匹配及关联数据.xlsx"         # Excel 文件路径
excel_file = f"C:/Users/User/Desktop/pythondata/预后模型评估/zs/output1增加的新的数据.xlsx"        # Excel 文件路径

sheet_name = "Sheet1"             # 工作表名称
      # 输出文件路径
col1 = "影像表现"                 # 第一列列名
col2 = "诊断结论"                   # 第二列列名


# 读取 Excel 数据
df = pd.read_excel(excel_file, sheet_name=sheet_name)
# with open("xmoutput.md", "r", encoding="utf-8") as file:
#     markdown_content = file.read()  


markdown_content =  """
下面是PET-CT报告的专业术语定义用来学习参考。

提取(FLs)影像表现、(EM)影像表现、(PM)影像表现、骨折影像表现、骨折诊断结论、(L)影像表现、(L)诊断结论、(FLs)诊断结论、(EM)诊断结论、(PM)诊断结论  （按出现顺序）。
局灶性病灶（FLs）:
局灶性病变部位分为S (Skull) SP (spine) Ex-Sp (extra-spine)。局灶性病变定义为在至少 2 个连续的 PET 切片上或大小大于5mm可见的 F-FDG 摄取增加的局灶区域，摄取增加指的是SUVmax值大于周围骨髓/骨骼组织或高于肝脏组织或大于2.5，需表现为局灶性（非弥漫性），即单个或多个离散的高摄取灶。需排除弥漫性摄取和生理学摄取（炎性摄取）。
髓外病灶(EM) :
髓外病变（EM）指的是软组织/器官高摄取灶。髓外病变（EM）数量分为N/EN(淋巴结内/淋巴结外)填写时应填写N或EN。
髓旁病灶(PM): 
髓旁病灶(PM)是指软组织肿块从骨髓生长到周围组织。髓外病变和髓旁病灶需要排除退行性病变、炎性病变以及非多发性骨髓瘤的肿瘤病变（影像表现和诊断结论作为依据）。
特别注意旁髓病变，临近骨且形成软组织肿块。
溶骨性病变(L):
溶骨性病变是指通过CT成像检测到的骨质破坏，表现为骨皮质或骨小梁的缺失，通常提示骨髓瘤细胞浸润导致的骨吸收和骨结构破坏。
骨折：
骨折指CT病理性骨折，骨折部位分为S (Skull) SP (spine) Ex-Sp (extra-spine)
使用原文进行提取，不要转述或重叠实体。

最终输出结构化表格及其示例共8个如下：

影像表现: 禁食状态下，静脉注射18F-FDG1h后行全身及脑显像。 大脑各部显像清晰，大脑皮质内放射性分布均匀，双侧顶叶、颞叶、枕叶及双侧基底节、双侧小脑放射性分布尚对称，未见明显放射性摄取增高或减低灶。CT平扫示余脑实质内未见异常密度灶，脑沟、脑裂、脑池未见增宽、扩张，其内密度如常，中线结构居中。 颈部显像清晰，PET／CT显像示鼻咽部、双侧颈部淋巴结分布区、双侧甲状腺等未见明显放射性摄取异常增高区及明确异常外形与密度改变。 胸部显像清晰，PET／CT显像示双肺纹理增多增粗紊乱，气管支气管通畅，双肺上叶及双肺下叶见多发稍高密度条索影、斑片影，放射性摄取未见增高；余肺野内未见放射性摄取异常增高灶；纵隔淋巴结和双侧肺门淋巴结未见明显增大及未见放射性摄取异常增高灶；食管走行区未见明显放射性摄取异常增高灶。 腹部显影清晰，PET／CT显像示肝脏形态饱满，放射性摄取未见增高；脾脏形态饱满伴放射性摄取轻度增高，SUV最大值约3.4；双肾见结节状钙化灶；胰腺及双侧肾上腺外形无明显异常，实质内未见异常密度灶及放射性摄取异常增高灶；胆囊形态不大，胆囊壁及胆囊腔未见异常密度影及异常放射性摄取增高影；双侧腹股沟淋巴结分布区均未见放射性摄取异常增高灶；胃腔充盈良好，胃壁未见明显增厚及异常放射性摄取增高灶；肠道走行及分布区放射性摄取浓淡不一，肠管形态结构无明确改变；前列腺、双侧精囊腺位置外形无明显异常，实质内未见异常密度灶及放射性摄取异常增高灶。 右侧额骨见明显骨质破坏影伴周围软组织肿胀，放射性摄取增高，范围约7.3*4.2cm，SUV最大值约10.1，病灶明显推挤右侧额叶；蝶骨偏左侧见明显骨质破坏影伴软组织密度团块影，放射性摄取增高，范围约4.6*3.4cm，SUV最大值约22.6。右侧锁骨、双侧肱骨、右侧肩胛骨、右（2、6）肋、骶骨、双侧髂骨、双侧股骨见多枚低密度骨质破坏影，周围见软组织密度肿块影并累及相邻肌群，范围大者位于左侧髂骨，范围约8.4*4.9cm，SUV最大值约23.1；右侧竖脊肌（约腰2-4椎体水平）明显增粗伴团块状软组织密度影伴放射性摄取增高，范围约7.4*6.6*8.8cm，SUV最大值约12.9；后腹膜区见软组织密度团块影伴放射性摄取增高，病灶包绕腹主动脉并与相邻双侧腰大肌分界不清，范围约7.0*7.6*13.1cm，SUV最大值约20.7；腹主动脉旁、双侧髂血管旁、双侧盆壁、双侧髂窝及骶前区另见数枚稍大淋巴结影伴放射性摄取增高，大者位于左侧盆壁，大小约1.2*1.5cm，SUV最大值约15.3；余扫描区骨骼骨质密度见多发稍低密度骨质破坏影，放射性摄取略增高，SUV最大值约4.4；腰2-骶1椎体呈术后改变。双侧多根肋骨见骨质密度增高影，放射性摄取未见增高；右侧胸大肌深面及双侧腋下见数枚稍大淋巴结影伴放射性摄取增高，大者位于右侧胸大肌深面，大小约1.0*0.8cm，SUV最大值约7.4；  	诊断结论：1、多发性骨髓瘤治疗后：扫描区多处骨骼（部位见上述）多发低密度骨质破坏影，周围见软组织密度肿块影并累及相邻肌群，FDG代谢增高，右侧竖脊肌（约腰2-4椎体水平）明显增粗伴团块状软组织密度影伴FDG代谢增高，后腹膜区见软组织密度团块影伴FDG代谢增高，病灶包绕腹主动脉并与相邻双侧腰大肌分界不清，右侧胸大肌深面及双侧腋下、腹主动脉旁、双侧髂血管旁、双侧盆壁、双侧髂窝及骶前区另见数枚稍大淋巴结影伴FDG代谢增高，考虑肿瘤多处（骨、肌肉、淋巴结）浸润伴高肿瘤活性存在，建议继续专科诊疗；双侧多根肋骨见骨质密度增高影，未见FDG代谢增，余扫描区骨骼骨质密度见多发稍低密度骨质破坏影，FDG代谢略高，建议密切随访；余全身（包括脑）PET显像未见FDG代谢明显异常增高灶。 2、双肺上叶及双肺下叶多发慢性炎症；双肾钙化灶；双侧肋骨多发陈旧性骨折影；腰2-骶1椎体呈术后改变。  	
|编号|部位|SUVmax|CT表现|
|---|---|---|---|
|(FLs)影像表现|右侧额骨|10.1|右侧额骨见明显骨质破坏影伴周围软组织肿胀，放射性摄取增高，范围约7.3*4.2cm，SUV最大值约10.1|
|(PM)影像表现|右侧锁骨、双侧肱骨、右侧肩胛骨、右（2、6）肋、骶骨、双侧髂骨、双侧股骨|23.1|右侧锁骨、双侧肱骨、右侧肩胛骨、右（2、6）肋、骶骨、双侧髂骨、双侧股骨见多枚低密度骨质破坏影，周围见软组织密度肿块影并累及相邻肌群，范围大者位于左侧髂骨，范围约8.4*4.9cm，SUV最大值约23.1|
|(FLs)影像表现|余扫描区骨骼|4.4|余扫描区骨骼骨质密度见多发稍低密度骨质破坏影，放射性摄取略增高，SUV最大值约4.4|
|(PM)影像表现|蝶骨偏左侧|22.6|蝶骨偏左侧见明显骨质破坏影伴软组织密度团块影，放射性摄取增高，范围约4.6*3.4cm，SUV最大值约22.6|
|(EM)影像表现|右侧竖脊肌|12.9|右侧竖脊肌（约腰2-4椎体水平）明显增粗伴团块状软组织密度影伴放射性摄取增高，范围约7.4*6.6*8.8cm，SUV最大值约12.9|
|(EM)影像表现|后腹膜区|20.7|后腹膜区见软组织密度团块影伴放射性摄取增高，病灶包绕腹主动脉并与相邻双侧腰大肌分界不清，范围约7.0*7.6*13.1cm，SUV最大值约20.7|
|(EM)影像表现|腹主动脉旁、双侧髂血管旁、双侧盆壁、双侧髂窝及骶前区|15.3|腹主动脉旁、双侧髂血管旁、双侧盆壁、双侧髂窝及骶前区另见数枚稍大淋巴结影伴放射性摄取增高，大者位于左侧盆壁，大小约1.2*1.5cm，SUV最大值约15.3|
|(EM)影像表现|右侧胸大肌深面及双侧腋下|7.4|右侧胸大肌深面及双侧腋下见数枚稍大淋巴结影伴放射性摄取增高，大者位于右侧胸大肌深面，大小约1.0*0.8cm，SUV最大值约7.4|
|(L)影像表现|右侧额骨|10.1|右侧额骨见明显骨质破坏影伴周围软组织肿胀，放射性摄取增高，范围约7.3*4.2cm，SUV最大值约10.1|
|(L)影像表现|蝶骨偏左侧|22.6|蝶骨偏左侧见明显骨质破坏影伴软组织密度团块影，放射性摄取增高，范围约4.6*3.4cm，SUV最大值约22.6|
|(L)影像表现|右侧锁骨、双侧肱骨、右侧肩胛骨、右（2、6）肋、骶骨、双侧髂骨、双侧股骨|23.1|右侧锁骨、双侧肱骨、右侧肩胛骨、右（2、6）肋、骶骨、双侧髂骨、双侧股骨见多枚低密度骨质破坏影，周围见软组织密度肿块影并累及相邻肌群，范围大者位于左侧髂骨，范围约8.4*4.9cm，SUV最大值约23.1|
|(L)影像表现|余扫描区骨骼|4.4|余扫描区骨骼骨质密度见多发稍低密度骨质破坏影，放射性摄取略增高，SUV最大值约4.4|
|骨折影像表现|双侧多根肋骨||双侧多根肋骨见骨质密度增高影，放射性摄取未见增高|
|(FLs)诊断结论|余扫描区骨骼||余扫描区骨骼骨质密度见多发稍低密度骨质破坏影，FDG代谢略高|
|(PM)诊断结论|扫描区多处骨骼||扫描区多处骨骼（部位见上述）多发低密度骨质破坏影，周围见软组织密度肿块影并累及相邻肌群，FDG代谢增高|
|(EM)诊断结论|右侧竖脊肌||右侧竖脊肌（约腰2-4椎体水平）明显增粗伴团块状软组织密度影伴FDG代谢增高|
|(EM)诊断结论|后腹膜区||后腹膜区见软组织密度团块影伴FDG代谢增高，病灶包绕腹主动脉并与相邻双侧腰大肌分界不清|
|(EM)诊断结论|右侧胸大肌深面及双侧腋下、腹主动脉旁、双侧髂血管旁、双侧盆壁、双侧髂窝及骶前区||右侧胸大肌深面及双侧腋下、腹主动脉旁、双侧髂血管旁、双侧盆壁、双侧髂窝及骶前区另见数枚稍大淋巴结影伴FDG代谢增高|
|(L)诊断结论|扫描区多处骨骼||扫描区多处骨骼（部位见上述）多发低密度骨质破坏影，周围见软组织密度肿块影并累及相邻肌群，FDG代谢增高|
|(L)诊断结论|余扫描区骨骼||余扫描区骨骼骨质密度见多发稍低密度骨质破坏影，FDG代谢略高|
|骨折诊断结论|双侧肋骨||双侧肋骨多发陈旧性骨折影|"

影像表现: 禁食状态下，静脉注射18F-FDG1h后行全身显像。 大脑各部显像清晰，大脑皮质内放射性分布均匀，双侧额叶、顶叶、颞叶、枕叶放射性分布尚对称，双侧基底节、双侧小脑放射性分布尚对称，未见明显放射性摄取增高或减低灶。CT平扫示左侧额叶见一低密度区，较大截面约2.0*2.4cm，放射性分布减低，邻近额骨左侧部分缺如；余脑实质内未见异常密度灶，脑沟、脑裂、脑池未见增宽、扩张，其内密度如常，中线结构居中。 颈部显像清晰，PET/CT显像示鼻咽部、余双侧颈部淋巴结分布区、双侧甲状腺等未见明显放射性摄取异常增高区及明确异常外形与密度改变。右鼻腔后部鼻中隔旁结节状放射性摄取增高，SUV最大值约7.5。 胸部显像清晰，PET/CT显像示双肺纹理清晰，气管支气管通畅，右肺水平裂软组织密度结节，大小约2.0*1.0cm，放射性摄取增高，SUV最大值约10.4，右肺中叶、左肺舌叶支气管扩张，双肺散在条索、斑片影，界尚清，放射性摄取未见增高，余肺野内未见放射性摄取异常增高灶；纵隔（2R、4R、5、6、7）及双肺门见数枚增大淋巴结显示，大者位于右肺门区，直径约1.2cm，放射性摄取增高，SUV最大值约6.6；食管走行区未见明显放射性摄取异常增高灶。双侧腋下小淋巴结影，未见放射性摄取异常增高灶。 腹部显影清晰，PET/CT显像示胰头部软组织密度结节，直径约1.4cm，放射性摄取增高，SUV最大值约15.5；肝脏、脾脏、双侧肾脏及双侧肾上腺、前列腺、双侧精囊腺位置外形无明显异常，实质内未见异常密度灶及放射性摄取异常增高灶；胆囊大小形态未见异常，腔内密度无殊；后腹膜、双侧腹股沟淋巴结分布区均未见放射性摄取异常增高灶，胃腔充盈良好，胃壁未见明显增厚及异常放射性摄取增高灶；肠道走行及分布区放射性摄取浓淡不一，肠管形态结构无明确改变。右侧肾后间隙、盆腔系膜间隙软组织密度结节，大者直径约2.3cm，放射性摄取增高，SUV最大值约10.3；左侧锁骨区、左胸廓入口处、食管中下段后方软组织密度结节，大者直径约1.0cm，放射性摄取增高，SUV最大值约12.6，右侧心膈角区软组织密度肿块，大小约5.9*3.6cm，放射性摄取增高，SUV最大值约17.8；左肾上极旁小结节状放射性摄取增高，SUV最大值约3.3。 双侧竖脊肌周围多发结节状放射性摄取增高，SUV最大值约14.3。 T2-5脊椎术后，T4-5椎体后方结节状放射性摄取增高，SUV最大值约16.4；左侧第2肋骨骨质破坏伴软组织密度肿块，大小约8.4*6.9cm，放射性摄取增高，SUV最大值约16.4，右8后肋骨质破坏伴病理性骨折及周围软组织增厚影，放射性摄取增高，SUV最大值约13.2；双侧胸膜多发结节状增厚伴放射性摄取增高，以右侧为著，SUV最大值约17.5，右肱骨近段髓腔局灶性放射性摄取稍增高，SUV最大值约2.9，余扫描区骨骼骨质密度不均匀性减低，放射性摄取欠均匀略增高，SUV最大值约3.8。  	诊断结论：1、多发性骨髓瘤治疗后，左侧第2肋骨骨质破坏伴周围较大软组织密度肿块，FDG代谢增高，右8后肋骨质破坏伴病理性骨折及周围软组织增厚影，双侧胸膜多发结节状增厚伴FDG代谢增高，以右侧为著，考虑治疗后高肿瘤活性存在（病灶较我院2021-10-8PET/CT增大增多），T2-5脊椎术后，T4-5椎体后方结节状FDG代谢增高，双侧竖脊肌周围多发结节状FDG代谢增高灶，左侧锁骨区、左胸廓入口处、食管中下段后方、右侧心膈角区、右侧肾后间隙、盆腔系膜间隙多发软组织密度影，胰头部软组织密度结节，右肺水平裂软组织密度结节，FDG代谢增高，均考虑肿瘤浸润；右肱骨近段髓腔局灶性FDG代谢稍增高，余扫描区骨骼骨质密度不均匀性减低，FDG代谢欠均匀略增高，请结合骨髓活检；纵隔多区及双肺门见数枚增大淋巴结显示，FDG代谢增高，建议密切随访；右鼻腔后部鼻中隔旁结节状FDG代谢增高，建议专科检查；左肾上极旁小结节状FDG代谢增高，建议密切随访；余全身（包括脑）PET显像未见FDG代谢明显异常增高灶。 2、左侧额叶斑片样低密度灶，邻近额骨左侧部分缺如，建议随访；右肺中叶、左肺舌叶支气管扩张；双肺散在炎性纤维灶。  	"
|编号|部位|SUVmax|CT表现|
|---|---|---|---|
|(FLs)影像表现|右肱骨近段|2.9|右肱骨近段髓腔局灶性放射性摄取稍增高，SUV最大值约2.9|
|(EM)影像表现|右肺水平裂|10.4|右肺水平裂软组织密度结节，大小约2.0*1.0cm，放射性摄取增高，SUV最大值约10.4|
|(EM)影像表现|纵隔及双肺门|6.6|纵隔（2R、4R、5、6、7）及双肺门见数枚增大淋巴结显示，大者位于右肺门区，直径约1.2cm，放射性摄取增高，SUV最大值约6.6|
|(EM)影像表现|胰头部|15.5|胰头部软组织密度结节，直径约1.4cm，放射性摄取增高，SUV最大值约15.5|
|(EM)影像表现|右侧肾后间隙、盆腔系膜间隙|10.3|右侧肾后间隙、盆腔系膜间隙软组织密度结节，大者直径约2.3cm，放射性摄取增高，SUV最大值约10.3|
|(EM)影像表现|左侧锁骨区、左胸廓入口处、食管中下段后方|12.6|左侧锁骨区、左胸廓入口处、食管中下段后方软组织密度结节，大者直径约1.0cm，放射性摄取增高，SUV最大值约12.6|
|(EM)影像表现|右侧心膈角区|17.8|右侧心膈角区软组织密度肿块，大小约5.9*3.6cm，放射性摄取增高，SUV最大值约17.8|
|(EM)影像表现|双侧竖脊肌周围|14.3|双侧竖脊肌周围多发结节状放射性摄取增高，SUV最大值约14.3|
|(PM)影像表现|左侧第2肋骨|16.4|左侧第2肋骨骨质破坏伴软组织密度肿块，大小约8.4*6.9cm，放射性摄取增高，SUV最大值约16.4|
|(EM)影像表现|双侧胸膜|17.5|双侧胸膜多发结节状增厚伴放射性摄取增高，以右侧为著，SUV最大值约17.5|
|骨折影像表现|右8后肋||右8后肋骨质破坏伴病理性骨折及周围软组织增厚影|
|骨折诊断结论|右8后肋||右8后肋骨质破坏伴病理性骨折及周围软组织增厚影|
|(L)影像表现|左侧第2肋骨||左侧第2肋骨骨质破坏伴软组织密度肿块|
|(L)影像表现|右8后肋||右8后肋骨质破坏伴病理性骨折及周围软组织增厚影|
|(L)诊断结论|左侧第2肋骨||左侧第2肋骨骨质破坏伴周围较大软组织密度肿块|
|(L)诊断结论|右8后肋||右8后肋骨质破坏伴病理性骨折及周围软组织增厚影|
|(FLs)诊断结论|右肱骨近段||右肱骨近段髓腔局灶性FDG代谢稍增高|
|(EM)诊断结论|右肺水平裂||右肺水平裂软组织密度结节，FDG代谢增高，均考虑肿瘤浸润|
|(EM)诊断结论|胰头部||胰头部软组织密度结节，FDG代谢增高，均考虑肿瘤浸润|
|(EM)诊断结论|右侧肾后间隙、盆腔系膜间隙||右侧肾后间隙、盆腔系膜间隙多发软组织密度影，FDG代谢增高，均考虑肿瘤浸润|
|(EM)诊断结论|左侧锁骨区、左胸廓入口处、食管中下段后方、右侧心膈角区、右侧肾后间隙、盆腔系膜间隙||左侧锁骨区、左胸廓入口处、食管中下段后方、右侧心膈角区、右侧肾后间隙、盆腔系膜间隙多发软组织密度影，FDG代谢增高，均考虑肿瘤浸润|
|(EM)诊断结论|双侧竖脊肌周围||双侧竖脊肌周围多发结节状FDG代谢增高灶，FDG代谢增高，均考虑肿瘤浸润|
|(EM)诊断结论|纵隔及双肺门||纵隔多区及双肺门见数枚增大淋巴结显示，FDG代谢增高，建议密切随访|
|(PM)诊断结论|左侧第2肋骨||左侧第2肋骨骨质破坏伴周围较大软组织密度肿块，FDG代谢增高|
|(EM)诊断结论|双侧胸膜||双侧胸膜多发结节状增厚伴FDG代谢增高，以右侧为著|"

影像表现: 禁食状态下，静脉注射18F-FDG1h后行全身及脑显像。 大脑各部显像清晰，大脑皮质内放射性分布均匀，双侧额叶、顶叶、颞叶、枕叶及双侧基底节、双侧小脑放射性分布尚对称，未见明显放射性摄取增高或减低灶。CT平扫示脑实质内未见异常密度灶，脑沟、脑裂、脑池未见增宽、扩张，其内密度如常，中线结构居中。 颈部显像清晰，PET/CT显像示右侧咽旁间隙、双侧腮腺区、双侧颈部、左侧锁骨区、双侧腋窝多发大小淋巴结，大者约0.7cm，放射性摄取增高，SUV最大值约3.2；鼻咽部度改变。 胸部显像清晰，PET/CT显像示双肺纹理清晰，气管支气管通畅，双侧胸膜下多发软组织密度结节，大者位于右肺上叶，大小约2.4*2.8cm，放射性摄取增高，SUV最大值约27.7。双肺散在条索影，未见摄取增高。余肺野内未见放射性摄取异常增高灶；双侧乳腺外形及密度无殊，腺体区未见放射性摄取异常增高灶。食管走行区未见明显放射性摄取异常增高灶。 腹部显影清晰，PET/CT显像示左肾盂至左侧输尿管上段见软组织密度影，放射性摄取增高，SUV最大值约8.8，左肾盂积液；左肾上腺增粗伴放射性摄取增高，SUV最大值约12.2；肝脏多发稍低密度结节，边界不清，大者位于右肝，大小约1.3*1.4cm，放射性摄取增高，SUV最大值约7.8；脾脏外形增大，未见摄取增高；胰腺、右侧肾脏及右侧肾上腺外形无明显异常，实质内未见异常密度灶及放射性摄取异常增高灶；右膈肌深面、双侧心膈角、后腹膜、腹盆腔肠系膜间隙、双侧盆壁旁多发稍大淋巴结，直径约1.1cm，轻度放射性摄取增高，SUV最大值约5.9；胆囊大小形态未见异常，胆囊壁及腔内未见异常密度影及异常放射性摄取增高灶；胃腔充盈良好，胃壁未见明显增厚及异常放射性摄取增高灶；肠道走行及分布区放射性摄取浓淡不一，肠管形态结构无明确改变；左侧附件小囊性密度结节，未见摄取增高；子宫及右侧附件区未见异常密度灶及放射性摄取异常增高灶。 扫描区骨骼（颅盖骨、颅底骨、双侧肱骨、双侧锁骨、胸骨、双侧多处肋骨、双侧肩胛骨、脊柱多处椎体、骨盆诸骨、两侧股骨）弥漫多发虫噬样骨质破坏伴不均匀放射性摄取增高，最大SUV值约16.8。双侧肩背部、胸壁、腹壁皮下脂肪多发小结节，大者约0.6cm，轻度放射性摄取增高，SUV最大值约2.4。右肩胛下肌、左侧斜方肌及左侧竖脊肌多发小结节样放射性摄取增高，SUV最大值约5.9。腰1、3椎体关节面下低密度结节，未见摄取增高。  	诊断结论：1、确诊多发性骨髓瘤治疗后：扫描区骨骼弥漫多发虫噬样骨质破坏伴不均匀FDG代谢增高，考虑多发性骨髓瘤治疗后仍伴高肿瘤活性存在；肝脏多发稍低密度结节，FDG代谢增高；左肾上腺增粗伴FDG代谢增高，双侧胸膜下多发软组织密度结节，FDG代谢增高，双侧肩背部、胸壁、腹壁皮下多发FDG代谢增高，右肩胛下肌、左侧斜方肌及左侧竖脊肌多发小结节样FDG代谢增高，以上，考虑多发浸润；扫描区（具体如上）多发稍大淋巴结伴FDG代谢增高，考虑多发淋巴结浸润；左肾盂至左侧输尿管上段见软组织密度影，FDG代谢增高，左肾盂积液，需鉴诊考虑肿瘤浸润或肾原发肿瘤可能，建议结合病理活检；余全身（包括脑）PET显像未见FDG代谢明显异常增高灶。 2、双肺支气管病变；左侧附件囊性结节，建议妇科随访；腰1、3椎体许莫氏结节。  	"
|编号|部位|SUVmax|CT表现|
|---|---|---|---|
|(FLs)影像表现|颅盖骨、颅底骨、双侧肱骨、双侧锁骨、胸骨、双侧多处肋骨、双侧肩胛骨、脊柱多处椎体、骨盆诸骨、两侧股骨||扫描区骨骼（颅盖骨、颅底骨、双侧肱骨、双侧锁骨、胸骨、双侧多处肋骨、双侧肩胛骨、脊柱多处椎体、骨盆诸骨、两侧股骨）弥漫多发虫噬样骨质破坏伴不均匀放射性摄取增高，最大SUV值约16.8。|
|(FLs)诊断结论|扫描区骨骼||扫描区骨骼弥漫多发虫噬样骨质破坏伴不均匀FDG代谢增高，考虑多发性骨髓瘤治疗后仍伴高肿瘤活性存在；|
|(EM)影像表现|双侧胸膜下|27.7|双侧胸膜下多发软组织密度结节，大者位于右肺上叶，大小约2.4*2.8cm，放射性摄取增高，SUV最大值约27.7。|
|(EM)影像表现|肝脏|7.8|肝脏多发稍低密度结节，边界不清，大者位于右肝，大小约1.3*1.4cm，放射性摄取增高，SUV最大值约7.8；|
|(EM)影像表现|左肾上腺|12.2|左肾上腺增粗伴放射性摄取增高，SUV最大值约12.2；|
|(EM)影像表现|双侧肩背部、胸壁、腹壁皮下|2.4|双侧肩背部、胸壁、腹壁皮下脂肪多发小结节，大者约0.6cm，轻度放射性摄取增高，SUV最大值约2.4。|
|(EM)影像表现|右肩胛下肌、左侧斜方肌、左侧竖脊肌|5.9|右肩胛下肌、左侧斜方肌及左侧竖脊肌多发小结节样放射性摄取增高，SUV最大值约5.9。|
|(EM)影像表现|右膈肌深面、双侧心膈角、后腹膜、腹盆腔肠系膜间隙、双侧盆壁旁|5.9|右膈肌深面、双侧心膈角、后腹膜、腹盆腔肠系膜间隙、双侧盆壁旁多发稍大淋巴结，直径约1.1cm，轻度放射性摄取增高，SUV最大值约5.9；|
|(EM)影像表现|右侧咽旁间隙、双侧腮腺区、双侧颈部、左侧锁骨区、双侧腋窝|3.2|右侧咽旁间隙、双侧腮腺区、双侧颈部、左侧锁骨区、双侧腋窝多发大小淋巴结，大者约0.7cm，放射性摄取增高，SUV最大值约3.2；|
|(EM)影像表现|左肾盂至左侧输尿管上段|8.8|左肾盂至左侧输尿管上段见软组织密度影，放射性摄取增高，SUV最大值约8.8，左肾盂积液；|
|(EM)诊断结论|肝脏、左肾上腺、双侧胸膜下、双侧肩背部胸壁腹壁皮下、右肩胛下肌、左侧斜方肌、左侧竖脊肌||肝脏多发稍低密度结节，FDG代谢增高；左肾上腺增粗伴FDG代谢增高，双侧胸膜下多发软组织密度结节，FDG代谢增高，双侧肩背部、胸壁、腹壁皮下多发FDG代谢增高，右肩胛下肌、左侧斜方肌及左侧竖脊肌多发小结节样FDG代谢增高，以上，考虑多发浸润；|
|(EM)诊断结论|扫描区多发淋巴结||扫描区（具体如上）多发稍大淋巴结伴FDG代谢增高，考虑多发淋巴结浸润；|
|(EM)诊断结论|左肾盂至左侧输尿管上段||左肾盂至左侧输尿管上段见软组织密度影，FDG代谢增高，左肾盂积液，需鉴诊考虑肿瘤浸润或肾原发肿瘤可能，建议结合病理活检；|
|(L)影像表现|颅盖骨、颅底骨、双侧肱骨、双侧锁骨、胸骨、双侧多处肋骨、双侧肩胛骨、脊柱多处椎体、骨盆诸骨、两侧股骨||扫描区骨骼（颅盖骨、颅底骨、双侧肱骨、双侧锁骨、胸骨、双侧多处肋骨、双侧肩胛骨、脊柱多处椎体、骨盆诸骨、两侧股骨）弥漫多发虫噬样骨质破坏伴不均匀放射性摄取增高，最大SUV值约16.8。|
|(L)诊断结论|扫描区骨骼||扫描区骨骼弥漫多发虫噬样骨质破坏伴不均匀FDG代谢增高，考虑多发性骨髓瘤治疗后仍伴高肿瘤活性存在；|"

影像表现: 禁食状态下，静脉注射18F-FDG1h后行全身及脑显像。 大脑各部显像清晰，大脑皮质内放射性分布均匀，双侧额叶、顶叶、颞叶、枕叶及双侧基底节、双侧小脑放射性分布尚对称，未见明显放射性摄取增高或减低灶。CT平扫示脑实质内未见异常密度灶，脑沟、脑裂、脑池未见增宽、扩张，其内密度如常，中线结构居中。 颈部显像清晰，PET／CT显像示右侧筛窦及上颌窦粘膜增厚，内见偏高密度影，放射性摄取增高，SUV最大值约7.7；两侧颈部、锁骨区散在淋巴结显示，大者位于右侧下颌角旁，径约0.7cm，放射性摄取增高，SUV最大值约5.9；鼻咽部、双侧甲状腺等未见明显放射性摄取异常增高区及明确异常外形与密度改变。 胸部显像清晰，PET／CT显像示双肺透亮度增加，双肺纹理增多，见散在少许条索影，气管支气管通畅，右肺上叶外段一枚类圆形透亮影，内无肺纹理，放射性摄取未见增高；余肺野内未见放射性摄取异常增高灶；纵隔淋巴结和双侧肺门淋巴结未见明显增大及未见放射性摄取异常增高灶；部分胃腔经食管裂孔进入胸腔，食管走行区未见明显放射性摄取异常增高灶；双侧腋下小淋巴结影，未见放射性摄取异常增高灶。 腹部显影清晰，PET／CT显像示肝脏饱满，实质密度弥漫性略减低，平均CT值小于50Hu，放射性摄取未见增高；胰腺、脾脏、双侧肾脏及双侧肾上腺外形无明显异常，实质内未见异常密度灶及放射性摄取异常增高灶；胆囊大小外形无殊，腔内见高密度结节影，放射性摄取未见增高；后腹膜、双侧腹股沟淋巴结分布区均未见放射性摄取异常增高灶；胃腔充盈良好，胃小弯侧条片状放射性摄取增高，SUV最大值约3.1；痔疮术后改变，术区未见明显异常密度影及放射性摄取异常增高；阑尾腔内密度增高，放射性摄取未见增高；余肠道走行及分布区放射性摄取浓淡不一，肠管形态结构无明确改变；前列腺不大，内见多发致密影，放射性摄取未见增高；双侧精囊腺位置外形无明显异常，实质内未见异常密度灶及放射性摄取异常增高灶。腹主动脉及其分支钙化斑块附着，放射性摄取未见增高。 扫描区骨骼（包括颅骨、双侧锁骨、肩胛骨、胸骨、双侧肋骨、脊柱椎体及附件、骨盆诸骨及扫描区四肢长骨近端）多发虫噬样低密度骨质改变，部分病灶伴软组织密度影，放射性摄取不均匀轻度增高，SUV最大值约6.6；T7-T9椎体变扁，T11-L1椎体内固定术后，局部见放射性摄取增高，SUV最大值约6.1；右肩胛骨骨骨折术后，术区内固定显示，局部放射性摄取片状增高，SUV最大值约6.3；。  	诊断结论：1、多发性骨髓瘤治疗后复查，扫描区骨骼（详见描述）多发虫噬样低密度骨质改变，部分病灶伴软组织密度影，FDG代谢不均匀轻度增高，对照本院2023-11-13 PET／CT检查，骨质破坏程度及病灶内代谢较前基本相仿，考虑治疗后仍伴肿瘤活性存在，建议专科继续治疗随访，必要时请结合骨髓活检；T7-T9椎体变扁，较前大致相仿，建议随诊复查；右肩胛骨骨骨折术后改变、T11-L1椎体内固定术后改变； 2、右侧筛窦及上颌窦粘膜增厚，内见偏高密度影，FDG代谢增高，较前新发，考虑鼻窦炎（炎症性可能），建议专科诊疗；两侧颈部、锁骨区散在淋巴结显示，FDG代谢增高，考虑炎性淋巴结，建议随访；胃小弯侧条片状FDG代谢增高，考虑炎性摄取，必要时请结合内镜检查；前片所示右肺下叶背段两枚磨玻璃结节本次PET-CT未见显示；余全身（包括脑）PET显像未见FDG代谢明显异常增高灶。 3、双肺支气管病变伴肺气肿，双肺散在少许纤维灶，右肺上叶肺气囊；食管裂孔疝伴炎症；脂肪肝；胆囊结石；痔疮术后改变；阑尾粪石；前列腺钙化灶；腹主动脉及其分支硬化。  	"
|编号|部位|SUVmax|CT表现|
|---|---|---|---|
|(FLs)影像表现|扫描区骨骼（包括颅骨、双侧锁骨、肩胛骨、胸骨、双侧肋骨、脊柱椎体及附件、骨盆诸骨及扫描区四肢长骨近端）|6.6|扫描区骨骼（包括颅骨、双侧锁骨、肩胛骨、胸骨、双侧肋骨、脊柱椎体及附件、骨盆诸骨及扫描区四肢长骨近端）多发虫噬样低密度骨质改变，部分病灶伴软组织密度影，放射性摄取不均匀轻度增高，SUV最大值约6.6；|
|(FLs)影像表现|右肩胛骨|6.3|右肩胛骨骨骨折术后，术区内固定显示，局部放射性摄取片状增高，SUV最大值约6.3；|
|(FLs)诊断结论|扫描区骨骼||扫描区骨骼（详见描述）多发虫噬样低密度骨质改变，部分病灶伴软组织密度影，FDG代谢不均匀轻度增高，对照本院2023-11-13 PET／CT检查，骨质破坏程度及病灶内代谢较前基本相仿，考虑治疗后仍伴肿瘤活性存在，建议专科继续治疗随访，必要时请结合骨髓活检；|
|(PM)影像表现|扫描区骨骼（包括颅骨、双侧锁骨、肩胛骨、胸骨、双侧肋骨、脊柱椎体及附件、骨盆诸骨及扫描区四肢长骨近端）|6.6|扫描区骨骼（包括颅骨、双侧锁骨、肩胛骨、胸骨、双侧肋骨、脊柱椎体及附件、骨盆诸骨及扫描区四肢长骨近端）多发虫噬样低密度骨质改变，部分病灶伴软组织密度影，放射性摄取不均匀轻度增高，SUV最大值约6.6；|
|(PM)诊断结论|扫描区骨骼||扫描区骨骼（详见描述）多发虫噬样低密度骨质改变，部分病灶伴软组织密度影，FDG代谢不均匀轻度增高，对照本院2023-11-13 PET／CT检查，骨质破坏程度及病灶内代谢较前基本相仿，考虑治疗后仍伴肿瘤活性存在，建议专科继续治疗随访，必要时请结合骨髓活检；|
|(L)影像表现|扫描区骨骼（包括颅骨、双侧锁骨、肩胛骨、胸骨、双侧肋骨、脊柱椎体及附件、骨盆诸骨及扫描区四肢长骨近端）|6.6|扫描区骨骼（包括颅骨、双侧锁骨、肩胛骨、胸骨、双侧肋骨、脊柱椎体及附件、骨盆诸骨及扫描区四肢长骨近端）多发虫噬样低密度骨质改变，部分病灶伴软组织密度影，放射性摄取不均匀轻度增高，SUV最大值约6.6；|
|(L)诊断结论|扫描区骨骼||扫描区骨骼（详见描述）多发虫噬样低密度骨质改变，部分病灶伴软组织密度影，FDG代谢不均匀轻度增高，对照本院2023-11-13 PET／CT检查，骨质破坏程度及病灶内代谢较前基本相仿，考虑治疗后仍伴肿瘤活性存在，建议专科继续治疗随访，必要时请结合骨髓活检；|
|骨折影像表现|T7-T9椎体、T11-L1椎体|6.1|T7-T9椎体变扁，T11-L1椎体内固定术后，局部见放射性摄取增高，SUV最大值约6.1；|
|骨折影像表现|右肩胛骨|6.3|右肩胛骨骨骨折术后，术区内固定显示，局部放射性摄取片状增高，SUV最大值约6.3；|
|骨折诊断结论|T7-T9椎体||T7-T9椎体变扁，较前大致相仿，建议随诊复查；|
|骨折诊断结论|右肩胛骨、T11-L1椎体||右肩胛骨骨骨折术后改变、T11-L1椎体内固定术后改变；|
|(EM)影像表现|两侧颈部、锁骨区|5.9|两侧颈部、锁骨区散在淋巴结显示，大者位于右侧下颌角旁，径约0.7cm，放射性摄取增高，SUV最大值约5.9；|
|(EM)诊断结论|两侧颈部、锁骨区||两侧颈部、锁骨区散在淋巴结显示，FDG代谢增高，考虑炎性淋巴结，建议随访；|"

影像表现: 禁食状态下，静脉注射^18^F-FDG1h后行全身显像。 大脑各部显像清晰，大脑皮质内放射性分布均匀，双侧额叶、顶叶、颞叶、枕叶放射性分布尚对称，双侧基底节、双侧小脑放射性分布尚对称，未见明显放射性摄取增高或减低灶。CT平扫示脑实质内未见异常密度灶，脑沟、脑裂、脑池未见增宽、扩张，其内密度如常，中线结构居中。 颈部显像清晰，PET/CT显像示双侧鼻咽部小片样放射性摄取增高，SUV最大值约为5.0；左侧颈根部淋巴结肿大，直径2.5cm，放射性摄取增高，SUV最大值约3.37；双侧甲状腺等未见明显放射性摄取异常增高区及明确异常外形与密度改变。 胸部显像清晰，PET/CT显像示右肺上叶尖段胸膜下见一枚磨玻璃密度小结节影，直径约0.6cm，放射性摄取未见增高；右肺上叶尖段及左肺下叶后基底段胸膜下见小结节影，放射性摄取未见增高；左肺上叶舌段及双肺下叶见少许条索影，放射性摄取未见增高；余双肺纹理增多，气管支气管通畅，余肺野内未见放射性摄取异常增高灶；纵隔内气管旁透亮影，纵隔淋巴结和双侧肺门淋巴结未见明显增大及未见放射性摄取异常增高灶；左侧乳腺区软组织密度团块影，大小6.0*3.8cm，边缘毛糙，放射性摄取增高，SUV最大值约3.7；右侧乳腺外形及密度未见异常，腺体区未见放射性摄取异常增高灶。双侧腋下小淋巴结影，放射性摄取未见异常增高；食管走行区未见明显放射性摄取异常增高灶。 腹部显影清晰，PET/CT显像示肝脏饱满，肝内见多发低密度影，界清，大者位于肝右叶近膈顶，大小约3.4*3.2cm，放射性摄取未见增高，余肝脏实质密度弥漫性略增高，SUV最大值约4.5；脾脏体积增大，约占6个肋单元，实质内未见异常密度影，放射性摄取弥漫性增高，SUV最大值约3.4；双肾见多发低密度影，界清，大者位于左肾，直径约1.6cm，放射性摄取未见增高；胰腺、双侧肾上腺结构位置外形无明显异常，实质内未见异常密度灶及放射性摄取异常增高灶；胆囊大小形态未见异常，腔内未见异常密度影，放射性摄取未见增高；后腹膜、双侧腹股沟淋巴结分布区均未见放射性摄取异常增高灶。胃腔充盈良好，胃壁未见明显增厚及异常放射性摄取增高灶；肠道走行及分布区放射性摄取浓淡不一，肠管形态结构无明确改变。盆腔内子宫体积小，子宫及双侧附件区未见异常密度灶及放射性摄取异常增高灶。 右侧肩部、右侧上臂肌群及皮下多发软组织密度团片影及结节影，大者长径6.1cm，放射性摄取增高，SUV最大值约7.35； 扫描区（颅骨、颅面骨、双侧肩胛骨、双侧锁骨、双肋、胸骨、颈胸腰椎椎体及附件、骨盆诸骨、双上肢及扫描区下肢）骨质密度不均匀性略减低，见散在低密度影，双侧股骨、右侧肱骨骨折，其中左侧股骨下段骨皮质骨质破坏伴软组织肿块影，放射性摄取增高，SUV最大值约7.2，右侧肱骨头及肱骨上段形态失常、骨皮质变薄，放射性摄取增高，SUV最大值约2.5，右侧股骨下段成角骨折及周围骨痂影，左侧第9-11肋见骨痂影，放射性摄取轻度增高，SUV最大值约5.3；扫描区骨髓腔内放射性摄取不均匀性增高，SUV最大值约3.8。  	诊断结论：1、多发性骨髓瘤治疗后，扫描区（详见描述）骨质密度不均匀减低，其中左侧股骨下段骨质破坏伴软组织肿块影，FDG代谢增高，右侧肱骨头及肱骨上段形态失常、骨皮质变薄，FDG代谢增高，双侧股骨、右侧肱骨、右侧股骨下段、左侧第9-11肋病理性骨折，FDG代谢轻度增高，与2022.2.16PET/CT比较，较前相仿，考虑治疗后伴肿瘤活性存在；右侧肩部、右侧上臂肌群及皮下多发软组织密度团片影及结节影，FDG代谢异常增高（新发病灶），左侧乳腺区软组织密度团块影（较前明显增大），FDG代谢异常增高，左侧颈根部淋巴结肿大（新发病灶），FDG代谢异常增高，考虑多发髓外浸润；扫描区骨髓腔内FDG代谢不均匀性增高，不除外伴肿瘤活性，建议结合骨髓活检随访；肝饱满，脾大，FDG代谢弥漫性增高，建议结合临床随访；余扫描所见区域全身（包括脑）PET显像未见FDG代谢明显异常增高灶。 2、双侧鼻咽部小片样FDG代谢增高，考虑炎症，建议随访；双侧颈部淋巴结反应性增生可能，建议随访；右肺上叶尖段胸膜下磨玻璃密度小结节影，FDG代谢未见增高，建议HRCT定期随访；双肺少许纤维灶及增殖灶；气管憩室；肝多发囊肿；双肾多发囊肿；老年性子宫。  	"
|编号|部位|SUVmax|CT表现|
|---|---|---|---|
|(FLs)影像表现|左侧股骨下段|7.2|其中左侧股骨下段骨皮质骨质破坏伴软组织肿块影，放射性摄取增高，SUV最大值约7.2|
|(FLs)影像表现|右侧肱骨头及肱骨上段|2.5|右侧肱骨头及肱骨上段形态失常、骨皮质变薄，放射性摄取增高，SUV最大值约2.5|
|(EM)影像表现|左侧颈根部|3.37|左侧颈根部淋巴结肿大，直径2.5cm，放射性摄取增高，SUV最大值约3.37|
|(EM)影像表现|左侧乳腺区|3.7|左侧乳腺区软组织密度团块影，大小6.0*3.8cm，边缘毛糙，放射性摄取增高，SUV最大值约3.7|
|(EM)影像表现|右侧肩部、右侧上臂肌群及皮下|7.35|右侧肩部、右侧上臂肌群及皮下多发软组织密度团片影及结节影，大者长径6.1cm，放射性摄取增高，SUV最大值约7.35|
|(PM)影像表现|左侧股骨下段|7.2|其中左侧股骨下段骨皮质骨质破坏伴软组织肿块影，放射性摄取增高，SUV最大值约7.2|
|骨折影像表现|双侧股骨、右侧肱骨||双侧股骨、右侧肱骨骨折|
|骨折影像表现|右侧股骨下段||右侧股骨下段成角骨折及周围骨痂影|
|骨折影像表现|左侧第9-11肋||左侧第9-11肋见骨痂影|
|骨折诊断结论|双侧股骨、右侧肱骨、右侧股骨下段、左侧第9-11肋||双侧股骨、右侧肱骨、右侧股骨下段、左侧第9-11肋病理性骨折，FDG代谢轻度增高|
|(L)影像表现|扫描区（颅骨、颅面骨、双侧肩胛骨、双侧锁骨、双肋、胸骨、颈胸腰椎椎体及附件、骨盆诸骨、双上肢及扫描区下肢）||扫描区（颅骨、颅面骨、双侧肩胛骨、双侧锁骨、双肋、胸骨、颈胸腰椎椎体及附件、骨盆诸骨、双上肢及扫描区下肢）骨质密度不均匀性略减低，见散在低密度影|
|(L)影像表现|左侧股骨下段|7.2|其中左侧股骨下段骨皮质骨质破坏伴软组织肿块影，放射性摄取增高，SUV最大值约7.2|
|(L)影像表现|右侧肱骨头及肱骨上段|2.5|右侧肱骨头及肱骨上段形态失常、骨皮质变薄，放射性摄取增高，SUV最大值约2.5|
|(L)诊断结论|扫描区（详见描述），左侧股骨下段，右侧肱骨头及肱骨上段||扫描区（详见描述）骨质密度不均匀减低，其中左侧股骨下段骨质破坏伴软组织肿块影，FDG代谢增高，右侧肱骨头及肱骨上段形态失常、骨皮质变薄，FDG代谢增高|
|(FLs)诊断结论|扫描区（详见描述），左侧股骨下段，右侧肱骨头及肱骨上段||扫描区（详见描述）骨质密度不均匀减低，其中左侧股骨下段骨质破坏伴软组织肿块影，FDG代谢增高，右侧肱骨头及肱骨上段形态失常、骨皮质变薄，FDG代谢增高|
|(EM)诊断结论|右侧肩部、右侧上臂肌群及皮下，左侧乳腺区，左侧颈根部||右侧肩部、右侧上臂肌群及皮下多发软组织密度团片影及结节影，FDG代谢异常增高（新发病灶），左侧乳腺区软组织密度团块影（较前明显增大），FDG代谢异常增高，左侧颈根部淋巴结肿大（新发病灶），FDG代谢异常增高，考虑多发髓外浸润|
|(PM)诊断结论|左侧股骨下段||其中左侧股骨下段骨质破坏伴软组织肿块影，FDG代谢增高|"

影像表现: 禁食状态下，静脉注射18F-FDG1h后行全身及脑显像。 大脑各部显像清晰，大脑皮质内放射性分布均匀，双侧额叶、顶叶、颞叶、枕叶放射性分布尚对称，双侧基底节、双侧小脑放射性分布尚对称，未见明显放射性摄取增高或减低灶。CT平扫示脑实质内未见异常密度灶，脑沟、脑裂、脑池未见增宽、扩张，其内密度如常，中线结构居中。 颈部显像清晰，PET／CT显像示双侧筛窦及右侧上颌窦附壁黏膜增厚，放射性摄取未见增高；左颈根部增大淋巴结影，直径约1.4cm，放射性摄取增高，SUV最大值约2.2；双侧扁桃腺放射性摄取增高，SUV最大值约16.4；双侧鼻咽部对称分布放射性摄取增高，SUV最大值约5.8，未见明显增厚；余双侧颈部淋巴结分布区、甲状腺等未见明显放射性摄取异常增高区及明确异常外形与密度改变。 右侧上臂肌群及皮下多发软组织密度团片影及结节影，较大截面约2.0*2.8cm，放射性摄取增高，SUV最大值约2.9； 胸部显像清晰，PET／CT显像示双肺纹理增多、增粗，气管支气管通畅，右肺上叶尖段可见一直径约0.8cm磨玻璃小结节影，界清，放射性摄取未见增高；右肺上叶尖段及左肺下叶后基底段胸膜下见细小结节影，放射性摄取未见增高；余双肺野内未见放射性摄取异常增高灶；纵隔淋巴结和双侧肺门淋巴结未见明显增大及未见放射性摄取异常增高灶；左侧乳腺区软组织密度结节、肿块影，大小病灶约3.2*4.4cm，其中伴低密度坏死区，边缘毛糙，放射性摄取增高，SUV最大值约6.0；右侧乳腺外形及密度无殊，腺体区未见放射性摄取异常增高灶。双侧腋下细小淋巴结显示，放射性摄取未见增高；食管走行区未见明显放射性摄取异常增高灶。 腹部显影清晰，PET／CT显像示肝脏外形未见异常，密度弥漫减低，平均CT密度约28hu，肝内多发囊状水样低密度影，较大一枚直径约3.2cm，界清，放射性摄取未见增高；脾脏未见增大，放射性摄取未见增高；子宫萎缩，双侧附件显示不清，放射性摄取未见增高；左侧肾上腺稍粗，放射性摄取轻度增高，SUV最大值约4.1；胰腺、双侧肾脏及右侧肾上腺外形无明显异常，实质内未见异常密度灶及放射性摄取异常增高灶；胆囊大小外形未见异常，胆囊壁及胆囊腔未见异常密度灶，放射性摄取未见增高；后腹膜、腹股沟淋巴结分布区均未见放射性摄取异常增高灶。胃腔充盈可，胃壁未见明显增厚及异常放射性摄取增高；肠道走行及分布区未见明显放射性摄取异常增高灶，肠管形态结构无明确改变。 扫描区骨骼骨质密度不均匀性略减低，见散在低密度影，右侧肱骨头及肱骨上段形态失常，双侧股骨、右侧肱骨骨折，其中左侧股骨下段骨皮质骨质破坏伴软组织肿块影，放射性摄取增高，SUV最大值约6.0；左侧第9-11肋局部骨皮质走行扭曲，放射性摄取未见增高；右肩关节及左髋关节放射性摄取增高，SUV最大值约6.4。  	诊断结论：1、多发性骨髓瘤治疗后，扫描区（详见描述）骨质密度不均匀减低，右侧肱骨头及肱骨上段形态失常，左侧股骨下段骨质破坏伴软组织肿块影，FDG代谢增高，对比2022-7-5日PET／CT左股骨下段软组织肿块影较前缩小，考虑治疗后伴仍有肿瘤活性存在；双侧股骨、右侧肱骨、左侧第9-11肋病理性骨折；右侧上臂肌群及皮下多发软组织密度团片影及结节影，FDG代谢增高，左侧乳腺区多枚软组织密度结节、肿块，FDG代谢增高，左侧颈根部淋巴结肿大，FDG代谢轻度增高，考虑多发髓外浸润，上述病灶均较前有所缩小，FDG代谢减低，考虑治疗后部分肿瘤活性受抑，仍有明显肿瘤组织存在，建议继续治疗后复查；左侧肾上腺增粗伴FDG代谢增高，考虑肾上腺增生可能，建议随访； 2、右肺上叶尖段胸膜下磨玻璃密度小结节影，FDG代谢未见增高，建议HRCT定期随访；余全身（包括脑）PET／CT显像未见FDG代谢异常增高灶。 3、双侧筛窦及右侧上颌窦少许慢性炎症；双侧鼻咽部及扁桃腺炎症；脂肪肝；肝多发囊肿；老年子宫。  	"
|编号|部位|SUVmax|CT表现|
|---|---|---|---|
|(EM)影像表现|左颈根部|2.2|左颈根部增大淋巴结影，直径约1.4cm，放射性摄取增高，SUV最大值约2.2；|
|(EM)影像表现|右侧上臂肌群及皮下|2.9|右侧上臂肌群及皮下多发软组织密度团片影及结节影，较大截面约2.0*2.8cm，放射性摄取增高，SUV最大值约2.9；|
|(EM)影像表现|左侧乳腺区|6.0|左侧乳腺区软组织密度结节、肿块影，大小病灶约3.2*4.4cm，其中伴低密度坏死区，边缘毛糙，放射性摄取增高，SUV最大值约6.0；|
|(PM)影像表现|左侧股骨下段|6.0|左侧股骨下段骨皮质骨质破坏伴软组织肿块影，放射性摄取增高，SUV最大值约6.0；|
|骨折影像表现|双侧股骨、右侧肱骨、左侧第9-11肋|6.0|双侧股骨、右侧肱骨骨折，其中左侧股骨下段骨皮质骨质破坏伴软组织肿块影，放射性摄取增高，SUV最大值约6.0；左侧第9-11肋局部骨皮质走行扭曲，放射性摄取未见增高；|
|骨折诊断结论|双侧股骨、右侧肱骨、左侧第9-11肋||双侧股骨、右侧肱骨、左侧第9-11肋病理性骨折；|
|(FLs)影像表现|扫描区骨骼|6.0|扫描区骨骼骨质密度不均匀性略减低，见散在低密度影，右侧肱骨头及肱骨上段形态失常，双侧股骨、右侧肱骨骨折，其中左侧股骨下段骨皮质骨质破坏伴软组织肿块影，放射性摄取增高，SUV最大值约6.0；|
|(L)影像表现|扫描区骨骼|6.0|扫描区骨骼骨质密度不均匀性略减低，见散在低密度影，右侧肱骨头及肱骨上段形态失常，双侧股骨、右侧肱骨骨折，其中左侧股骨下段骨皮质骨质破坏伴软组织肿块影，放射性摄取增高，SUV最大值约6.0；|
|(FLs)诊断结论|扫描区骨骼||扫描区（详见描述）骨质密度不均匀减低，右侧肱骨头及肱骨上段形态失常，左侧股骨下段骨质破坏伴软组织肿块影，FDG代谢增高，对比2022-7-5日PET／CT左股骨下段软组织肿块影较前缩小，考虑治疗后伴仍有肿瘤活性存在；|
|(L)诊断结论|扫描区骨骼||扫描区（详见描述）骨质密度不均匀减低，右侧肱骨头及肱骨上段形态失常，左侧股骨下段骨质破坏伴软组织肿块影，FDG代谢增高，对比2022-7-5日PET／CT左股骨下段软组织肿块影较前缩小，考虑治疗后伴仍有肿瘤活性存在；|
|(EM)诊断结论|右侧上臂肌群及皮下、左侧乳腺区、左侧颈根部||右侧上臂肌群及皮下多发软组织密度团片影及结节影，FDG代谢增高，左侧乳腺区多枚软组织密度结节、肿块，FDG代谢增高，左侧颈根部淋巴结肿大，FDG代谢轻度增高，考虑多发髓外浸润，上述病灶均较前有所缩小，FDG代谢减低，考虑治疗后部分肿瘤活性受抑，仍有明显肿瘤组织存在，建议继续治疗后复查；|
|(PM)诊断结论|左侧股骨下段||左侧股骨下段骨质破坏伴软组织肿块影，FDG代谢增高，对比2022-7-5日PET／CT左股骨下段软组织肿块影较前缩小，考虑治疗后伴仍有肿瘤活性存在；|"

影像表现: 禁食状态下，静脉注射18F-FDG 1h后行全身及脑显像。 大脑各部显像清晰，CT平扫示脑实质内未见异常密度灶，脑沟、脑裂、脑池未见增宽、扩张，其内密度如常，中线结构居中。大脑皮质内放射性分布均匀，双侧额叶、顶叶、颞叶、枕叶及双侧基底节、双侧小脑放射性分布尚对称，未见明显放射性摄取增高或减低灶。 颈部显像清晰，PET／CT显像示鼻咽部、双侧颈部淋巴结分布区、双侧甲状腺等未见异常外形与密度改变及放射性摄取异常增高区。 胸部显像清晰，PET／CT显像示双肺纹理增粗增多，气管支气管通畅，双肺多发小结节状高密度影，较大者直径约0.4cm，未见放射性摄取；余肺野内未见放射性摄取异常增高灶；食管走行条状放射性摄取增高，SUV最大值约7.9；双侧乳腺外形无殊，腺体致密，右乳散在斑点状致密影，未见放射性摄取异常增高灶。左锁骨区、纵隔、双侧肺门、右侧腋下小淋巴结影，纵隔7区较大者约0.7*1.0cm，放射性摄取略增高，最大SUV值约3.8。 腹部显影清晰，PET／CT显像示左侧肾上腺稍低密度结节，大小约1.4*1.7cm，边界清，放射性摄取略增高，最大SUV值约4.1；肝脏、胰腺、脾脏、双侧肾脏及右侧肾上腺外形无明显异常，实质内未见异常密度灶及放射性摄取异常增高灶；胆囊术后缺如；胃腔充盈良好，胃壁未见明显增厚及异常放射性摄取增高灶；升结肠、乙状结肠条状放射性摄取增高，SUV最大值约11.8，余肠道走行及分布区放射性摄取浓淡不一，肠管形态结构无明确改变；子宫及双侧附件区未见异常密度灶及放射性摄取异常增高灶；后腹膜、双侧腹股沟淋巴结分布区均未见放射性摄取异常增高灶。双侧臀部皮下钙化灶。 脊椎骨边缘变锐利；扫描区骨骼（颅骨、胸骨、双侧锁骨、肩胛骨、肋骨、脊柱椎体及附件、骨盆诸骨及四肢近段）多发骨质密度减低伴骨质破坏，局部伴软组织密度影，其中左5肋、右2肋、右6肋呈病理性骨折，放射性摄取增高，SUV最大值约7.5。  	诊断结论：1、多发性骨髓瘤化疗后：扫描区骨骼（颅骨、胸骨、双侧锁骨、肩胛骨、肋骨、脊柱椎体及附件、骨盆诸骨及四肢近段）多发骨质密度减低伴骨质破坏，局部伴软组织密度影，其中左5肋、右2肋、右6肋呈病理性骨折，FDG代谢增高，对比2021-07-16PET／CT，骨质破坏较前增多、代谢增高，考虑肿瘤活性存在； 2、食管走行条状FDG代谢增高，升结肠、乙状结肠条状FDG代谢增高，考虑炎症，建议内镜随访；左侧肾上腺稍低密度结节，FDG代谢略增高，考虑腺瘤可能，建议随访；余全身（包括脑）PET显像未见FDG代谢明显异常增高灶。 3、双肺支气管病变；双肺增殖钙化灶；双侧乳腺增生，右乳钙化灶；左锁骨区、纵隔、双侧肺门、右侧腋下淋巴结反应性增生；胆囊术后缺如；双侧臀部皮下钙化灶；椎体退变。  	"
|编号|部位|SUVmax|CT表现|
|---|---|---|---|
|(FLs)影像表现|扫描区骨骼（颅骨、胸骨、双侧锁骨、肩胛骨、肋骨、脊柱椎体及附件、骨盆诸骨及四肢近段）|7.5|扫描区骨骼（颅骨、胸骨、双侧锁骨、肩胛骨、肋骨、脊柱椎体及附件、骨盆诸骨及四肢近段）多发骨质密度减低伴骨质破坏，局部伴软组织密度影，其中左5肋、右2肋、右6肋呈病理性骨折，放射性摄取增高，SUV最大值约7.5。||
(PM)影像表现|扫描区骨骼（颅骨、胸骨、双侧锁骨、肩胛骨、肋骨、脊柱椎体及附件、骨盆诸骨及四肢近段）|7.5|扫描区骨骼（颅骨、胸骨、双侧锁骨、肩胛骨、肋骨、脊柱椎体及附件、骨盆诸骨及四肢近段）多发骨质密度减低伴骨质破坏，局部伴软组织密度影，其中左5肋、右2肋、右6肋呈病理性骨折，放射性摄取增高，SUV最大值约7.5。|
|骨折影像表现|左5肋、右2肋、右6肋|7.5|其中左5肋、右2肋、右6肋呈病理性骨折，放射性摄取增高，SUV最大值约7.5。|
|骨折诊断结论|左5肋、右2肋、右6肋||其中左5肋、右2肋、右6肋呈病理性骨折，FDG代谢增高，|
|(L)影像表现|扫描区骨骼（颅骨、胸骨、双侧锁骨、肩胛骨、肋骨、脊柱椎体及附件、骨盆诸骨及四肢近段）|7.5|扫描区骨骼（颅骨、胸骨、双侧锁骨、肩胛骨、肋骨、脊柱椎体及附件、骨盆诸骨及四肢近段）多发骨质密度减低伴骨质破坏，局部伴软组织密度影，其中左5肋、右2肋、右6肋呈病理性骨折，放射性摄取增高，SUV最大值约7.5。|
|(L)诊断结论|扫描区骨骼（颅骨、胸骨、双侧锁骨、肩胛骨、肋骨、脊柱椎体及附件、骨盆诸骨及四肢近段）||扫描区骨骼（颅骨、胸骨、双侧锁骨、肩胛骨、肋骨、脊柱椎体及附件、骨盆诸骨及四肢近段）多发骨质密度减低伴骨质破坏，局部伴软组织密度影，其中左5肋、右2肋、右6肋呈病理性骨折，FDG代谢增高，对比2021-07-16PET／CT，骨质破坏较前增多、代谢增高，考虑肿瘤活性存在；|
|(FLs)诊断结论|扫描区骨骼（颅骨、胸骨、双侧锁骨、肩胛骨、肋骨、脊柱椎体及附件、骨盆诸骨及四肢近段）||扫描区骨骼（颅骨、胸骨、双侧锁骨、肩胛骨、肋骨、脊柱椎体及附件、骨盆诸骨及四肢近段）多发骨质密度减低伴骨质破坏，局部伴软组织密度影，其中左5肋、右2肋、右6肋呈病理性骨折，FDG代谢增高，对比2021-07-16PET／CT，骨质破坏较前增多、代谢增高，考虑肿瘤活性存在；||
(PM)诊断结论|扫描区骨骼（颅骨、胸骨、双侧锁骨、肩胛骨、肋骨、脊柱椎体及附件、骨盆诸骨及四肢近段）||扫描区骨骼（颅骨、胸骨、双侧锁骨、肩胛骨、肋骨、脊柱椎体及附件、骨盆诸骨及四肢近段）多发骨质密度减低伴骨质破坏，局部伴软组织密度影，其中左5肋、右2肋、右6肋呈病理性骨折，FDG代谢增高，对比2021-07-16PET／CT，骨质破坏较前增多、代谢增高，考虑肿瘤活性存在；|"

影像表现: 禁食状态下，静脉注射18F-FDG1h后行全身及脑显像。 大脑各部显像清晰，大脑皮质内放射性分布均匀，双侧顶叶、颞叶、枕叶及双侧基底节、双侧小脑放射性分布尚对称，未见明显放射性摄取增高或减低灶。CT平扫示余脑实质内未见异常密度灶，脑沟、脑裂、脑池未见增宽、扩张，其内密度如常，中线结构居中。 颈部显像清晰，PET／CT显像示鼻咽部、双侧颈部淋巴结分布区、双侧甲状腺等未见明显放射性摄取异常增高区及明确异常外形与密度改变。 胸部显像清晰，PET／CT显像示双肺纹理增多增粗紊乱，气管支气管通畅，双肺上叶及双肺下叶见多发稍高密度条索影、斑片影，放射性摄取未见增高；余肺野内未见放射性摄取异常增高灶；纵隔淋巴结和双侧肺门淋巴结未见明显增大及未见放射性摄取异常增高灶；食管走行区未见明显放射性摄取异常增高灶。 腹部显影清晰，PET／CT显像示肝脏形态饱满，放射性摄取未见增高；脾脏形态饱满伴放射性摄取轻度增高，SUV最大值约3.4；双肾见结节状钙化灶；胰腺及双侧肾上腺外形无明显异常，实质内未见异常密度灶及放射性摄取异常增高灶；胆囊形态不大，胆囊壁及胆囊腔未见异常密度影及异常放射性摄取增高影；双侧腹股沟淋巴结分布区均未见放射性摄取异常增高灶；胃腔充盈良好，胃壁未见明显增厚及异常放射性摄取增高灶；肠道走行及分布区放射性摄取浓淡不一，肠管形态结构无明确改变；前列腺、双侧精囊腺位置外形无明显异常，实质内未见异常密度灶及放射性摄取异常增高灶。 右侧额骨见明显骨质破坏影伴周围软组织肿胀，放射性摄取增高，范围约7.3*4.2cm，SUV最大值约10.1，病灶明显推挤右侧额叶；蝶骨偏左侧见明显骨质破坏影伴软组织密度团块影，放射性摄取增高，范围约4.6*3.4cm，SUV最大值约22.6。右侧锁骨、双侧肱骨、右侧肩胛骨、右（2、6）肋、骶骨、双侧髂骨、双侧股骨见多枚低密度骨质破坏影，周围见软组织密度肿块影并累及相邻肌群，范围大者位于左侧髂骨，范围约8.4*4.9cm，SUV最大值约23.1；右侧竖脊肌（约腰2-4椎体水平）明显增粗伴团块状软组织密度影伴放射性摄取增高，范围约7.4*6.6*8.8cm，SUV最大值约12.9；后腹膜区见软组织密度团块影伴放射性摄取增高，病灶包绕腹主动脉并与相邻双侧腰大肌分界不清，范围约7.0*7.6*13.1cm，SUV最大值约20.7；腹主动脉旁、双侧髂血管旁、双侧盆壁、双侧髂窝及骶前区另见数枚稍大淋巴结影伴放射性摄取增高，大者位于左侧盆壁，大小约1.2*1.5cm，SUV最大值约15.3；余扫描区骨骼骨质密度见多发稍低密度骨质破坏影，放射性摄取略增高，SUV最大值约4.4；腰2-骶1椎体呈术后改变。双侧多根肋骨见骨质密度增高影，放射性摄取未见增高；右侧胸大肌深面及双侧腋下见数枚稍大淋巴结影伴放射性摄取增高，大者位于右侧胸大肌深面，大小约1.0*0.8cm，SUV最大值约7.4；  	诊断结论：1、多发性骨髓瘤治疗后：扫描区多处骨骼（部位见上述）多发低密度骨质破坏影，周围见软组织密度肿块影并累及相邻肌群，FDG代谢增高，右侧竖脊肌（约腰2-4椎体水平）明显增粗伴团块状软组织密度影伴FDG代谢增高，后腹膜区见软组织密度团块影伴FDG代谢增高，病灶包绕腹主动脉并与相邻双侧腰大肌分界不清，右侧胸大肌深面及双侧腋下、腹主动脉旁、双侧髂血管旁、双侧盆壁、双侧髂窝及骶前区另见数枚稍大淋巴结影伴FDG代谢增高，考虑肿瘤多处（骨、肌肉、淋巴结）浸润伴高肿瘤活性存在，建议继续专科诊疗；双侧多根肋骨见骨质密度增高影，未见FDG代谢增，余扫描区骨骼骨质密度见多发稍低密度骨质破坏影，FDG代谢略高，建议密切随访；余全身（包括脑）PET显像未见FDG代谢明显异常增高灶。 2、双肺上叶及双肺下叶多发慢性炎症；双肾钙化灶；双侧肋骨多发陈旧性骨折影；腰2-骶1椎体呈术后改变。  	"
|编号|部位|SUVmax|CT表现|
|---|---|---|---|
|(FLs)影像表现|右侧额骨|10.1|右侧额骨见明显骨质破坏影伴周围软组织肿胀，放射性摄取增高，范围约7.3*4.2cm，SUV最大值约10.1|
|(PM)影像表现|右侧锁骨、双侧肱骨、右侧肩胛骨、右（2、6）肋、骶骨、双侧髂骨、双侧股骨|23.1|右侧锁骨、双侧肱骨、右侧肩胛骨、右（2、6）肋、骶骨、双侧髂骨、双侧股骨见多枚低密度骨质破坏影，周围见软组织密度肿块影并累及相邻肌群，范围大者位于左侧髂骨，范围约8.4*4.9cm，SUV最大值约23.1|
|(FLs)影像表现|余扫描区骨骼|4.4|余扫描区骨骼骨质密度见多发稍低密度骨质破坏影，放射性摄取略增高，SUV最大值约4.4|
|(PM)影像表现|蝶骨偏左侧|22.6|蝶骨偏左侧见明显骨质破坏影伴软组织密度团块影，放射性摄取增高，范围约4.6*3.4cm，SUV最大值约22.6|
|(EM)影像表现|右侧竖脊肌|12.9|右侧竖脊肌（约腰2-4椎体水平）明显增粗伴团块状软组织密度影伴放射性摄取增高，范围约7.4*6.6*8.8cm，SUV最大值约12.9|
|(EM)影像表现|后腹膜区|20.7|后腹膜区见软组织密度团块影伴放射性摄取增高，病灶包绕腹主动脉并与相邻双侧腰大肌分界不清，范围约7.0*7.6*13.1cm，SUV最大值约20.7|
|(EM)影像表现|腹主动脉旁、双侧髂血管旁、双侧盆壁、双侧髂窝及骶前区|15.3|腹主动脉旁、双侧髂血管旁、双侧盆壁、双侧髂窝及骶前区另见数枚稍大淋巴结影伴放射性摄取增高，大者位于左侧盆壁，大小约1.2*1.5cm，SUV最大值约15.3|
|(EM)影像表现|右侧胸大肌深面及双侧腋下|7.4|右侧胸大肌深面及双侧腋下见数枚稍大淋巴结影伴放射性摄取增高，大者位于右侧胸大肌深面，大小约1.0*0.8cm，SUV最大值约7.4|
|(L)影像表现|右侧额骨|10.1|右侧额骨见明显骨质破坏影伴周围软组织肿胀，放射性摄取增高，范围约7.3*4.2cm，SUV最大值约10.1|
|(L)影像表现|蝶骨偏左侧|22.6|蝶骨偏左侧见明显骨质破坏影伴软组织密度团块影，放射性摄取增高，范围约4.6*3.4cm，SUV最大值约22.6|
|(L)影像表现|右侧锁骨、双侧肱骨、右侧肩胛骨、右（2、6）肋、骶骨、双侧髂骨、双侧股骨|23.1|右侧锁骨、双侧肱骨、右侧肩胛骨、右（2、6）肋、骶骨、双侧髂骨、双侧股骨见多枚低密度骨质破坏影，周围见软组织密度肿块影并累及相邻肌群，范围大者位于左侧髂骨，范围约8.4*4.9cm，SUV最大值约23.1|
|(L)影像表现|余扫描区骨骼|4.4|余扫描区骨骼骨质密度见多发稍低密度骨质破坏影，放射性摄取略增高，SUV最大值约4.4|
|骨折影像表现|双侧多根肋骨||双侧多根肋骨见骨质密度增高影，放射性摄取未见增高|
|(FLs)诊断结论|余扫描区骨骼||余扫描区骨骼骨质密度见多发稍低密度骨质破坏影，FDG代谢略高|
|(PM)诊断结论|扫描区多处骨骼||扫描区多处骨骼（部位见上述）多发低密度骨质破坏影，周围见软组织密度肿块影并累及相邻肌群，FDG代谢增高|
|(EM)诊断结论|右侧竖脊肌||右侧竖脊肌（约腰2-4椎体水平）明显增粗伴团块状软组织密度影伴FDG代谢增高|
|(EM)诊断结论|后腹膜区||后腹膜区见软组织密度团块影伴FDG代谢增高，病灶包绕腹主动脉并与相邻双侧腰大肌分界不清|
|(EM)诊断结论|右侧胸大肌深面及双侧腋下、腹主动脉旁、双侧髂血管旁、双侧盆壁、双侧髂窝及骶前区||右侧胸大肌深面及双侧腋下、腹主动脉旁、双侧髂血管旁、双侧盆壁、双侧髂窝及骶前区另见数枚稍大淋巴结影伴FDG代谢增高|
|(L)诊断结论|扫描区多处骨骼||扫描区多处骨骼（部位见上述）多发低密度骨质破坏影，周围见软组织密度肿块影并累及相邻肌群，FDG代谢增高|
|(L)诊断结论|余扫描区骨骼||余扫描区骨骼骨质密度见多发稍低密度骨质破坏影，FDG代谢略高|
|骨折诊断结论|双侧肋骨||双侧肋骨多发陈旧性骨折影|"


"""
# 生成 JSON 数据
# def process_row(row):
#     return f"{delimiter1}{row[col1]}{delimiter2}{row[col2]}" + "请生成对应的且格式一致的示例输出表格。"
mark =  "你是放射科专家，擅长进行PET-CT报告解读。"
json_obj = []
for idx, row in df.iterrows():
        # 拼接两列内容
    user_content = f"PET-CT报告如下\n" + f"影像表现：{row[col1]}  诊断结论：{row[col2]}" + "多发性骨髓瘤PET-CT自由报告如上所述" + "/n我的请求是将结构化表格（待填写）部分填写完整，并严格按照其格式输出，仅输出示例格式的结构化表格。列格式 |编号|部位|SUVmax|CT表现|"
        # 构建 JSON 对象
    json_obj.append({
        "custom_id": str(idx + 1),
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "qwen3",
            "messages":[
                {"role": "user", "content": mark + markdown_content + user_content}
            ],"temperature": 0.7, "top_p": 0.9}
    })
        
        # 写入文件（JSON Lines 格式）
    with open(r"C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\外部验证集评测\qwen34bthinkinglang表格输入prompt.jsonl", "w", encoding="utf-8") as f:    
        for item in json_obj:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"生成完成，文件已保存至 ")

生成完成，文件已保存至 


In [ ]:

#deepseek qwenplus
import pandas as pd
import json
import re



            
# 配置参数



# 读取 Excel 数据

# with open("xmoutput.md", "r", encoding="utf-8") as file:
#     markdown_content = file.read()  
markdown_content =  """
下面是初诊基线评估的术语定义用来学习参考。

骨髓浸润与骨质破坏模式分类四类，分别为1.Minimal (normal appearing) 2.Focal lesions 3. Diffuse infiltration and bone destruction 4.Mixed (focal lesions on diffuse background)
骨髓浸润与骨质破坏模式 填写规范：Minimal/Focal lesions/Diffuse infiltration and bone destruction/Mixed  示例：Minimal

局灶性病灶绝对数目三类：0、1–3、>3
局灶性病灶绝对数目 填写规范：0/1–3/>3

骨折情况：需判断新增还是陈旧，骨折部位以及良恶性。
骨折情况 填写规范：文本（文本格式 新增/陈旧，部位文本，良性/恶性）示例：新增,T7，恶性

髓外病变：有 / 无（软组织/淋巴结/器官高代谢）
髓外病变 填写规范： 有/无

髓旁病变：有 / 无（自骨髓向外生长的软组织肿块）
髓旁病变 填写规范： 有/无

长骨浸润：仅指长骨，浸润指代谢弥漫性增高或有局限性病灶。有 / 无（股骨、肱骨等）
长骨浸润 填写规范： 有/无

骨骼手术证据：有 / 无（既往手术痕迹或金属植入）
骨骼手术证据 填写规范： 有/无


最终输出初诊基线评估结构化表格3行7列（待填写）如下：
| 骨髓浸润与骨质破坏模式   | 局灶性病灶绝对数目 |  骨折情况      |     髓外病变        |      髓旁病变      |    长骨浸润        |   骨骼手术证据   |
|------------------------|-----------------|----------------|------------------|--------------------|---------------------|----------------|
|  （待填写）             |  （待填写）      |     （待填写）   |   （待填写）      |    （待填写）      |  （待填写）          |（待填写）      |

真实示例：
| 骨髓浸润与骨质破坏模式   | 局灶性病灶绝对数目 | 骨折情况 | 髓外病变 | 髓旁病变 | 长骨浸润 | 骨骼手术证据 |
|------------------------|------------------|----------|----------|----------|-----------|--------------|
|  Diffuse infiltration and bone destruction    | 0               | 无       | 无       | 无       | 有        | 无           |

| 骨髓浸润与骨质破坏模式    | 局灶性病灶绝对数目 | 骨折情况                  | 髓外病变 | 髓旁病变 | 长骨浸润 | 骨骼手术证据 |
|---------------------|-------------------|---------------------------|----------|----------|----------|--------------|
| Focal lesions       | >3                 | 新增,左7肋及左10肋,恶性   | 无       | 有       | 有       | 无           |

"""
# 生成 JSON 数据
# def process_row(row):
#     return f"{delimiter1}{row[col1]}{delimiter2}{row[col2]}" + "请生成对应的且格式一致的示例输出表格。"
json_obj = []
col1 = "影像表现"                 # 第一列列名
col2 = "诊断结论"                   # 第二列列名


df_excel = pd.read_excel(f"C:/Users/User/Desktop/pythondata/预后模型评估/zs/output1增加的新的数据.xlsx")
with open(r'C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\loraqwen34bthinkingzenjiashuju结构化表格输出.jsonl', 'r', encoding='utf-8') as f:
    for idx, line in enumerate(f):
        if idx > 111: 
            break
        excel_row = df_excel.iloc[idx]
        jsonl_row = json.loads(line.strip())
#for idx, row in df_excel.iterrows():
        # 拼接两列内容['body']['choices'][0]['message']['content']    病灶明细表：{excel_row['病灶明细表'] if not pd.isna(excel_row['病灶明细表']) else '无病灶'}
        user_content = f"PET-CT报告如下\n" + f"影像表现：{excel_row[col1]}  诊断结论：{excel_row[col2]}" + "多发性骨髓瘤PET-CT自由报告如上所述" +"你是血液科医生"+f"PET-CT结构化表格：{jsonl_row['response'].split('</think>')[-1].strip()}  " + "初诊基线评估表格生成所需的多发性骨髓瘤PET-CT报告的结构化表格如上，请将初诊基线评估表格（待填写）部分填写完整，并严格按照其格式输出，最终仅输出初诊基线评估表格。"
        # 构建 JSON 对象
        json_obj.append({
            "custom_id": str(idx + 1),
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "1",
                "messages":[
                    {"role": "user", "content": markdown_content + user_content}
                ],"temperature": 0, "top_p": 0.9}
        })
        
        # 写入文件（JSON Lines 格式）
        with open("loraqwen34bthinkingzenjiashuju初诊基线表输入prompt.jsonl", "w", encoding="utf-8") as f:    
            for item in json_obj:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"生成完成，文件已保存至 ")

生成完成，文件已保存至 


In [ ]:

#deepseek qwenplus
import pandas as pd
import json
import re




# 读取 Excel 数据

# with open("xmoutput.md", "r", encoding="utf-8") as file:
#     markdown_content = file.read()  
markdown_content =  """
下面是评估的术语定义用来学习参考。

MRD评估：
MRD阴性指的是必须同时满足下列所有影像学要求
骨髓所有原先受累区域骨髓信号/代谢恢复正常，无任何弥漫或局灶高代谢
局灶性病变完全消失：PET-CT 局灶计数 = 0，且无任何新发病灶
所有软组织肿瘤（paramedullary & extramedullary）完全消失
既往未受累区域不得出现任何新浸润或病灶

以上条件都满足
注意：MRD 阴性的“金标准”定义
• 必须 先满足骨髓 MRD 阴性（NGF 或 NGS，敏感度 ≥10⁻⁵）
• 且所有基线 PET-CT 阳性病灶完全消失，或 SUV 低于 纵隔血池或周围正常组织
• 建议 间隔 ≥1 年重复骨髓 + 影像均为阴性，方可称为“持续影像学 MRD 阴性

MRD阴性是否 填写规范： 是/否

最终仅输出是或否
"""
# 生成 JSON 数据
# def process_row(row):
#     return f"{delimiter1}{row[col1]}{delimiter2}{row[col2]}" + "请生成对应的且格式一致的示例输出表格。"
json_obj = []
col1 = "影像表现"                 # 第一列列名
col2 = "诊断结论"                   # 第二列列名

df_excel = pd.read_excel(f"C:/Users/User/Desktop/pythondata/预后模型评估/zs/output1增加的新的数据.xlsx")
with open(r'C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\loraqwen34bthinkingzenjiashuju结构化表格输出.jsonl', 'r', encoding='utf-8') as f:
    for idx, line in enumerate(f):
        if idx < 112:
            continue
        excel_row = df_excel.iloc[idx]
        jsonl_row = json.loads(line.strip())
#for idx, row in df_excel.iterrows():
        # 拼接两列内容['body']['choices'][0]['message']['content']
        user_content = f"PET-CT报告如下\n" + f"影像表现：{excel_row[col1]}  诊断结论：{excel_row[col2]}" + "多发性骨髓瘤PET-CT自由报告如上所述"+"你是血液科医生"+f"PET-CT结构化表格：{jsonl_row['response'].split('</think>')[-1].strip()}  " + "最终仅输出是或否"
        # 构建 JSON 对象
        json_obj.append({
            "custom_id": str(idx+1 ),
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "",
                "messages":[
                    {"role": "user", "content": markdown_content + user_content}
                ],"temperature": 0, "top_p": 0.9}
        })
        
        # 写入文件（JSON Lines 格式）
        with open(r"loraqwen34bthinkingzenjiashujumrd输入.jsonl", "w", encoding="utf-8") as f:    
            for item in json_obj:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"生成完成，文件已保存至 ")

生成完成，文件已保存至 


In [10]:
import pandas as pd
import numpy as np

# 假设你的Excel有以下列
# 姓名, 病历号, 治疗阶段(1=治疗前, 2=治疗后, 3=复发), 其他临床数据...
#df = pd.read_excel("C:/Users/User/Desktop/pythondata/预后模型评估/10112/病灶明细表与结构化表格初诊基线MRD金标准_cleaned_fixed copy.xlsx")
df = pd.read_excel("C:/Users/User/Desktop/pythondata/预后模型评估/zs/output1增加的新的数据.xlsx")

# 1. 新增评估类型列


# 2. 创建匹配键（优先用病历号，病历号缺失则用姓名）
def make_match_key(row):
    if pd.notna(row['病历号']) and str(row['病历号']).strip() != '':
        return f"{row['姓名']}|{row['病历号']}"
    else:
        return f"{row['姓名']}|_MISSING_"

df['match_key'] = df.apply(make_match_key, axis=1)

# 3. 分离不同阶段的数据
df_pre = df[df['治疗前后/复发'] == 1].copy()  # 治疗前
df_post = df[df['治疗前后/复发'] == 2].copy()  # 治疗后
df_relapse = df[df['治疗前后/复发'] == 3].copy()  # 复发

# 4. 为治疗后和复发记录找到对应的治疗前记录
# 构建治疗前的匹配键到索引的映射
pre_key_to_idx = df_pre.groupby('match_key').apply(lambda x: x.index[0]).to_dict()

# 为治疗后记录匹配治疗前
df_post['对应治疗前索引'] = df_post['match_key'].map(pre_key_to_idx)

# 为复发记录匹配治疗前
df_relapse['对应治疗前索引'] = df_relapse['match_key'].map(pre_key_to_idx)

# 5. 合并回原DataFrame
df['对应治疗前索引'] = np.nan
df.loc[df_post.index, '对应治疗前索引'] = df_post['对应治疗前索引']
df.loc[df_relapse.index, '对应治疗前索引'] = df_relapse['对应治疗前索引']

# 6. 新增匹配状态列
df['是否匹配到治疗前'] = df['对应治疗前索引'].notna()
# 7. 可选：新增匹配阶段列，明确是治疗后还是复发
#df['匹配阶段'] = df['治疗阶段'].map({2: '治疗后', 3: '复发'})


# 8. 保存结果
df.to_excel(r"C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\suifangsuoying.xlsx", index=False)

# 9. 验证配对结果
print("配对统计：")
print(df.groupby(['治疗前后/复发', '是否匹配到治疗前']).size())
print("\n治疗后配对详情：")
for idx, row in df[df['治疗前后/复发'] == 2].iterrows():
    print(f"治疗后记录 {idx}: {row['姓名']}, 病历号={row['病历号']}, "
          f"匹配治疗前索引: {row['对应治疗前索引']}, "
          f"匹配成功: {row['是否匹配到治疗前']}")

print("\n复发配对详情：")
for idx, row in df[df['治疗前后/复发'] == 3].iterrows():
    print(f"复发记录 {idx}: {row['姓名']}, 病历号={row['病历号']}, "
          f"匹配治疗前索引: {row['对应治疗前索引']}, "
          f"匹配成功: {row['是否匹配到治疗前']}")

C:\Users\User\AppData\Local\Temp\ipykernel_10620\2772918408.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pre_key_to_idx = df_pre.groupby('match_key').apply(lambda x: x.index[0]).to_dict()


配对统计：
治疗前后/复发  是否匹配到治疗前
1        False       112
2        False        11
         True         14
3        False         5
         True         21
dtype: int64

治疗后配对详情：
治疗后记录 112: 方勇英, 病历号=3999510, 匹配治疗前索引: 0.0, 匹配成功: True
治疗后记录 113: 方勇英, 病历号=3999510, 匹配治疗前索引: 0.0, 匹配成功: True
治疗后记录 114: 徐加来, 病历号=2300033680, 匹配治疗前索引: 30.0, 匹配成功: True
治疗后记录 115: 谢宝娥, 病历号=2201029205, 匹配治疗前索引: 3.0, 匹配成功: True
治疗后记录 116: 桂积渊, 病历号=2300465104, 匹配治疗前索引: 6.0, 匹配成功: True
治疗后记录 117: 徐寿根, 病历号=2201609849, 匹配治疗前索引: 19.0, 匹配成功: True
治疗后记录 118: 曾步英, 病历号=2300399746, 匹配治疗前索引: nan, 匹配成功: False
治疗后记录 119: 颜建如, 病历号=3108474, 匹配治疗前索引: 33.0, 匹配成功: True
治疗后记录 120: 韩南生, 病历号=2300092075, 匹配治疗前索引: 35.0, 匹配成功: True
治疗后记录 121: 黄双凤, 病历号=2300272657, 匹配治疗前索引: nan, 匹配成功: False
治疗后记录 122: 杨利斌, 病历号=2300278285, 匹配治疗前索引: nan, 匹配成功: False
治疗后记录 123: 李玲丽, 病历号=2300694480, 匹配治疗前索引: 53.0, 匹配成功: True
治疗后记录 124: 沈平, 病历号=826737, 匹配治疗前索引: 57.0, 匹配成功: True
治疗后记录 125: 张广丽, 病历号=2300598865, 匹配治疗前索引: nan, 匹配成功: False
治疗后记录 126: 祝长泉, 病历号=2201102958, 匹配

In [4]:
import pandas as pd
import json
pd.set_option('display.max_colwidth', None)  # 或设置一个较大的值
pd.set_option('display.max_rows', None)      # 显示所有行

df_excel_og = pd.read_excel(r"C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\suifangsuoying.xlsx",sheet_name='Sheet2')


data = []
with open(r'C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\loraqwen34bthinkingzenjiashuju结构化表格输出.jsonl', 'r', encoding='utf-8') as f:
    for idx,line in enumerate(f):
        line = line.strip()
        if line:  # 跳过空行
            try:
                json_obj = json.loads(line)
                data.append(json_obj)
            except json.JSONDecodeError as e:
                print(f"第{idx+1}行JSON解析错误: {e}")
                print(f"错误行内容: {line[:100]}...")
df_jghbg = pd.DataFrame(data) 

data=[]
with open(r'C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\loraqwen34bthinkingzenjiashujumrd输出.jsonl', 'r', encoding='utf-8') as f:
    for idx,line in enumerate(f):
        line = line.strip()
        if line:  # 跳过空行
            try:
                json_obj = json.loads(line)
                data.append(json_obj)
            except json.JSONDecodeError as e:
                print(f"第{idx+1}行JSON解析错误: {e}")
                print(f"错误行内容: {line[:100]}...")
df_mrd = pd.DataFrame(data) 

data=[]
with open(r'C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\loraqwen34bthinkingzenjiashuju初诊基线表输出.jsonl', 'r', encoding='utf-8') as f:
    for idx,line in enumerate(f):
        line = line.strip()
        if line:  # 跳过空行
            try:
                json_obj = json.loads(line)
                data.append(json_obj)
            except json.JSONDecodeError as e:
                print(f"第{idx+1}行JSON解析错误: {e}")
                print(f"错误行内容: {line[:100]}...")
df_czjx = pd.DataFrame(data) 

markdown_content =  """
下面是随访疗效评估的术语定义用来学习参考。

随访疗效评估：
Response（缓解）指的是必须同时满足下列所有影像学要求
• Normalisation of bone marrow signal in previously affected areas
• Decrease in the number and size of focal lesions
• Resolution of severely infiltrated bone marrow infiltrate into focal lesions
• Decrease in the of number and size of soft tissue tumors (paramedullary and extramedullary)
Progression（进展）指的是只要出现以下任一项即可判定
• Worsening of diffuse bone marrow signal or new appearance of infiltration in previously unaffected areas
• Increase in the number and size of focal lesions
• Merging of focal lesions into severely infiltrated bone marrow
• Increase in the size or number of soft tissue tumours(paramedullary and extramedullary)
No change（稳定）指的是上述各项均无明显增减


"""

json_obj = []
     
df_excel_suifang = pd.read_excel(r'C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\suifangsuoying.xlsx', sheet_name='Sheet1')
for idx, row in df_excel_suifang.iterrows():
    lineidx = row["对应治疗前索引"]
    lineidx0 = row["PD_索引(从0开始)"]
    qianjghbg = df_jghbg.loc[df_jghbg['custom_id'] == str(lineidx+1),"response"].iloc[0].split('</think>')[-1].strip()
    benjghbg = df_jghbg.loc[df_jghbg['custom_id'] == str(lineidx0+1),"response"].iloc[0].split('</think>')[-1].strip()
    benmrd = df_mrd.loc[df_mrd['custom_id'] == str(lineidx0+1),"response"].iloc[0].split('</think>')[-1].strip()
    qianczjx = df_czjx.loc[df_czjx['custom_id'] == str(lineidx+1),"response"].iloc[0].split('</think>')[-1].strip()
    qianyxbx = df_excel_og.loc[lineidx]['影像表现']
    qianzdjl = df_excel_og.loc[lineidx]['诊断结论']
    benyxbx = row['影像表现']
    
    user_content = f"对应治疗前PET-CT报告如下\n" + f"对应治疗前影像表现：{qianyxbx}\n\n  对应治疗前诊断结论：{qianzdjl}\n\n" + f"AI 辅助生成的治疗前信息如下"+f"对应治疗前初诊基线表{qianczjx}\n 对应治疗前结构化表格{qianjghbg} \n"+ f"治疗后PET-CT报告如下\n" + f"治疗后影像表现：{row['影像表现']}\n\n  治疗后诊断结论：{row['诊断结论']}\n\n" + f"AI 辅助生成的治疗后信息如下" + f"治疗后mrd{benmrd}阴性\n 治疗后结构化表格{benjghbg} \n"+ f"两份多发性骨髓瘤PET-CT自由报告及相关AI辅助信息（仅参考）如上所述" + f"\n我的请求是随访疗效评估最终仅输出 稳定 缓解 进展 三个选项的其中一个"
        # 构建 JSON 对象
    json_obj.append({
        "custom_id": str(idx + 1),
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "qwen3",
            "messages":[
                {"role": "user", "content": markdown_content + user_content}
            ],"temperature": 0.7, "top_p": 0.9}
    })
        
        # 写入文件（JSON Lines 格式）
with open(r"C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\loraqwen34bthinkingzenjiashuju随访输入prompt.jsonl", "w", encoding="utf-8") as f:    
    for item in json_obj:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

    
    
    

jsonl2xlsx


In [7]:

# 2. 初始化一个列表来存储数据
import json
import re
import pandas as pd

# 文件路径
jsonl_file = r'C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\loraqwen34bthinkingzenjiashuju结构化表格输出.jsonl'
excel_file =  f"C:/Users/User/Desktop/pythondata/预后模型评估/zs/output1增加的新的数据.xlsx"# 输出的Excel文件名
sheet_name = "Sheet1"
target_column = "结构化表格thinking"

# 读取 JSONL 文件并提取 content
contents = []
with open(jsonl_file, "r", encoding="utf-8") as f:
    for line in f:
        try:
            data = json.loads(line.strip())
            # 提取 content 内容
           # content = data.get("response", {}).get("body", {}).get("choices", [{}])[0].get("message", {}).get("content", "")
            content = data.get("response", {}).split('</think>')[0].strip()
            # 截断 |\n\n 之后的内容
            processed_content = re.split(r'\n\n', content)[0]
            contents.append(content)
        except Exception as e:
            print(f"解析 JSON 行出错：{e}")
            contents.append("")

# 读取 Excel 文件
try:
    df = pd.read_excel(excel_file, sheet_name=sheet_name)
except FileNotFoundError:
    # 若文件不存在，创建一个空 DataFrame
    df = pd.DataFrame()

# 如果 deep0seek-md 列不存在，先创建该列
if target_column not in df.columns:
    df[target_column] = None

# 确保内容长度不超过现有行数，否则扩展行
max_len = max(len(contents), len(df))
df = df.reindex(range(max_len))

# 将处理后的内容按顺序写入 deepseek-md 列
for i, content in enumerate(contents):
    if i < len(df):
        df.at[i, target_column] = content
    else:
        df.loc[i, target_column] = content

# 保存回 Excel 文件
df.to_excel(excel_file, index=False, sheet_name=sheet_name, engine='openpyxl')

print(f"处理完成，结果已追加至 {excel_file} 的 '{target_column}' 列")

处理完成，结果已追加至 C:/Users/User/Desktop/pythondata/预后模型评估/zs/output1增加的新的数据.xlsx 的 '结构化表格thinking' 列


In [8]:

# 2. 初始化一个列表来存储数据
import json
import re
import pandas as pd

# 文件路径
jsonl_file = r'C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\loraqwen34bthinkingzenjiashuju初诊基线表输出.jsonl'
excel_file =  f"C:/Users/User/Desktop/pythondata/预后模型评估/zs/output1增加的新的数据.xlsx"# 输出的Excel文件名
sheet_name = "Sheet1"
target_column = "初诊基线表格表格thinking"

# 读取 JSONL 文件并提取 content
contents = []
with open(jsonl_file, "r", encoding="utf-8") as f:
    for line in f:
        try:
            data = json.loads(line.strip())
            # 提取 content 内容
           # content = data.get("response", {}).get("body", {}).get("choices", [{}])[0].get("message", {}).get("content", "")
            content = data.get("response", {}).split('</think>')[0].strip()
            # 截断 |\n\n 之后的内容
            processed_content = re.split(r'\n\n', content)[0]
            contents.append(content)
        except Exception as e:
            print(f"解析 JSON 行出错：{e}")
            contents.append("")

# 读取 Excel 文件
try:
    df = pd.read_excel(excel_file, sheet_name=sheet_name)
except FileNotFoundError:
    # 若文件不存在，创建一个空 DataFrame
    df = pd.DataFrame()

# 如果 deep0seek-md 列不存在，先创建该列
if target_column not in df.columns:
    df[target_column] = None

# 确保内容长度不超过现有行数，否则扩展行
max_len = max(len(contents), len(df))
df = df.reindex(range(max_len))

# 将处理后的内容按顺序写入 deepseek-md 列
for i, content in enumerate(contents):
    if i < len(df):
        df.at[i, target_column] = content
    else:
        df.loc[i, target_column] = content

# 保存回 Excel 文件
df.to_excel(excel_file, index=False, sheet_name=sheet_name, engine='openpyxl')

print(f"处理完成，结果已追加至 {excel_file} 的 '{target_column}' 列")

处理完成，结果已追加至 C:/Users/User/Desktop/pythondata/预后模型评估/zs/output1增加的新的数据.xlsx 的 '初诊基线表格表格thinking' 列


In [9]:
import json
import re
import pandas as pd

# 文件路径
jsonl_file = r'C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\loraqwen34bthinkingzenjiashujumrd输出.jsonl'
excel_file = r"C:/Users/User/Desktop/pythondata/预后模型评估/zs/output1增加的新的数据.xlsx"
sheet_name = "Sheet1"
target_column = "mrd表格thinking"

# 定义偏移量：JSONL 的前 112 行跳过，对应 Excel 索引 112 (即第 113 行)
OFFSET = 112

# 1. 读取 JSONL 文件并提取 content
contents = []
with open(jsonl_file, "r", encoding="utf-8") as f:
    for idx,line in enumerate(f):
        try:         
            data = json.loads(line.strip())
            # 提取 content 内容
            # 注意：这里假设 data.get("response") 返回的是字符串
            raw_response = data.get("response", "")
            
            if isinstance(raw_response, str):
                # 分割 </think> 并取最后一部分
                content = raw_response.split('</think>')[0].strip()
                # 截断 \n\n 之后的内容
                processed_content = re.split(r'\n\n', content)[0]
            else:
                processed_content = ""
                
            contents.append(processed_content)
        except Exception as e:
            print(f"解析 JSON 行出错 (索引 {idx})：{e}")
            contents.append("")

print(f"成功提取了 {len(contents)} 条数据。")

# 2. 读取 Excel 文件
try:
    df = pd.read_excel(excel_file, sheet_name=sheet_name)
    print(f"Excel 原有行数：{len(df)}")
except FileNotFoundError:
    print(f"警告：找不到 Excel 文件，将创建新文件。")
    df = pd.DataFrame()
except Exception as e:
    print(f"读取 Excel 失败：{e}")
    exit()

# 3. 确保列存在
if target_column not in df.columns:
    df[target_column] = None

# 4. 动态扩展 Excel 行数
# 计算需要的总行数：偏移量 + 新数据的数量
required_length = OFFSET + len(contents)

if len(df) < required_length:
    print(f"正在扩展 Excel 行数从 {len(df)} 到 {required_length}...")
    # 使用 reindex 扩展 DataFrame，新增的行会自动填充 NaN
    df = df.reindex(range(required_length))

# 5. 将处理后的内容写入指定位置
print("正在写入数据...")
for i, content in enumerate(contents):
    # 计算在 Excel 中的实际行索引
    excel_idx = OFFSET + i
    
    # 写入数据
    df.at[excel_idx, target_column] = content

# 6. 保存回 Excel 文件
try:
    df.to_excel(excel_file, index=False, sheet_name=sheet_name, engine='openpyxl')
    print(f"处理完成！结果已保存至 {excel_file}")
    print(f"数据已填充到 '{target_column}' 列，从第 {OFFSET + 1} 行开始。")
except Exception as e:
    print(f"保存 Excel 失败：{e}")

成功提取了 51 条数据。
Excel 原有行数：163
正在写入数据...
处理完成！结果已保存至 C:/Users/User/Desktop/pythondata/预后模型评估/zs/output1增加的新的数据.xlsx
数据已填充到 'mrd表格thinking' 列，从第 113 行开始。


In [6]:

# 2. 初始化一个列表来存储数据
import json
import re
import pandas as pd

# 文件路径
jsonl_file = r'C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\外部验证集评测\qwen34bthinkinglang表格输出.jsonl'
excel_file =  r"C:\Users\User\Desktop\pythondata\预后模型评估\zs\output1增加的新的数据.xlsx"# 输出的Excel文件名
sheet_name = "Sheet1"
target_column = "病灶明细表"

# 读取 JSONL 文件并提取 content
contents = []
with open(jsonl_file, "r", encoding="utf-8") as f:
    for line in f:
        try:
            data = json.loads(line.strip())
            # 提取 content 内容
           # content = data.get("response", {}).get("body", {}).get("choices", [{}])[0].get("message", {}).get("content", "")
            content = data.get("response", {}).split('</think>')[1].strip()
            # 截断 |\n\n 之后的内容
            processed_content = re.split(r'\n\n', content)[0]
            contents.append(content)
        except Exception as e:
            print(f"解析 JSON 行出错：{e}")
            contents.append("")

# 读取 Excel 文件
try:
    df = pd.read_excel(excel_file, sheet_name=sheet_name)
except FileNotFoundError:
    # 若文件不存在，创建一个空 DataFrame
    df = pd.DataFrame()

# 如果 deep0seek-md 列不存在，先创建该列
if target_column not in df.columns:
    df[target_column] = None

# 确保内容长度不超过现有行数，否则扩展行
max_len = max(len(contents), len(df))
df = df.reindex(range(max_len))

# 将处理后的内容按顺序写入 deepseek-md 列
for i, content in enumerate(contents):
    if i < len(df):
        df.at[i, target_column] = content
    else:
        df.loc[i, target_column] = content

# 保存回 Excel 文件
df.to_excel(excel_file, index=False, sheet_name=sheet_name, engine='openpyxl')

print(f"处理完成，结果已追加至 {excel_file} 的 '{target_column}' 列")

处理完成，结果已追加至 C:\Users\User\Desktop\pythondata\预后模型评估\zs\output1增加的新的数据.xlsx 的 '病灶明细表' 列


In [ ]:

# 2. 初始化一个列表来存储数据
import json
import re
import pandas as pd

# 文件路径
jsonl_file = r'C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\外部验证集评测\qwen34bthinkinglang表格输出.jsonl'
excel_file =  r"C:\Users\User\Desktop\pythondata\预后模型评估\10112\bidui\jiegouh\多模型评测\结构化表格\随访结果.xlsx"# 输出的Excel文件名
sheet_name = "Sheet1"
target_column = "随访thinking"

# 读取 JSONL 文件并提取 content
contents = []
with open(jsonl_file, "r", encoding="utf-8") as f:
    for line in f:
        try:
            data = json.loads(line.strip())
            # 提取 content 内容
           # content = data.get("response", {}).get("body", {}).get("choices", [{}])[0].get("message", {}).get("content", "")
            content = data.get("response", {}).split('</think>')[0].strip()
            # 截断 |\n\n 之后的内容
            processed_content = re.split(r'\n\n', content)[0]
            contents.append(content)
        except Exception as e:
            print(f"解析 JSON 行出错：{e}")
            contents.append("")

# 读取 Excel 文件
try:
    df = pd.read_excel(excel_file, sheet_name=sheet_name)
except FileNotFoundError:
    # 若文件不存在，创建一个空 DataFrame
    df = pd.DataFrame()

# 如果 deep0seek-md 列不存在，先创建该列
if target_column not in df.columns:
    df[target_column] = None

# 确保内容长度不超过现有行数，否则扩展行
max_len = max(len(contents), len(df))
df = df.reindex(range(max_len))

# 将处理后的内容按顺序写入 deepseek-md 列
for i, content in enumerate(contents):
    if i < len(df):
        df.at[i, target_column] = content
    else:
        df.loc[i, target_column] = content

# 保存回 Excel 文件
df.to_excel(excel_file, index=False, sheet_name=sheet_name, engine='openpyxl')

print(f"处理完成，结果已追加至 {excel_file} 的 '{target_column}' 列")